# MandiaCompact S2S — étape 6 : réponses à des dialogues écrits

Deux continuations du même modèle Mandia de 13 782 528 paramètres : CE de référence et CE + objectif contrastif sur la source. 800 mises à jour chacune ; mêmes lots et passages négatifs dans les deux essais. Aucun poids préentraîné externe.

**Données :** 160 exemples, 8 documents de SMOL. Cibles baoulé publiées sous CC BY 4.0 ; français traduit/adapté de l’anglais par l’assistant. Scénarios écrits/générés, pas des conversations naturelles enregistrées. Aucune validation native Mandia. Documents de validation et test séparés ; chevauchements exacts contrôlés avec les étapes précédentes. La Bible complète et les audios ne sont pas ajoutés.

**Utilisation :** sélectionner un GPU L4 ou compatible bf16, exécuter la préparation puis l’entraînement. Le meilleur modèle de l’étape 5 et les données de l’étape 4 doivent être présents dans le même Drive. Tous les poids, états AdamW/RNG et rapports vont directement dans un nouveau dossier Drive. Les sauvegardes antérieures sont conservées. Reprise possible en relançant les mêmes cellules avec les mêmes sources.

**Protocole :** sélection sur validation uniquement, avec contrôle de conservation des 14 anciennes réponses apprises et des pertes de validation précédentes. Les poids de chaque essai restent distincts ; aucune supériorité présumée du contraste. Les résultats test ne sont calculés qu’après enregistrement de la sélection. Coût supplémentaire du passage négatif compté dans les deux bras, même si le poids contrastif vaut zéro pour le témoin.

Sources : https://huggingface.co/datasets/google/smol ; https://arxiv.org/abs/2502.12301 . Code Mandia Apache-2.0 ; crédits et provenance dans le dossier Drive.


## Résultat du run exécuté le 14 octobre 2026

Deux entraînements de **800 étapes** terminés sur NVIDIA L4. Sélection sur validation : **CE, étape 100**. Les checkpoints de reprise à 800 et les meilleurs poids à 100 sont distincts et conservés dans Drive.

| Mesure test | Avant | CE retenue | Contraste |
|---|---:|---:|---:|
| chrF++ traduction, 25 exemples | 10.742 | 13.186 | 12.561 |
| chrF++ réponse, 15 exemples | 11.728 | 12.704 | 12.945 |
| Perte ancien test étape 4 | 3.621 | 3.663 | 3.666 |
| Perte ancien test étape 5 | 4.146 | 4.195 | 4.197 |

**Conclusion :** gains automatiques modestes sur le nouveau test, légère régression sur les anciens tests. Le contraste n’améliore pas la sélection. Les erreurs de sens restent nombreuses ; candidat de recherche, sans validation native. 0/25 traductions et 0/15 réponses exactement identiques aux références. Les 14 réponses simples apprises restent correctes.

**Données :** 160 vues préparées (110 traductions, 50 réponses), 157 après contrôle de longueur. Les vues partagent des cibles ; les deux tâches test viennent de seulement 2 documents. Français traduit/adapté par assistant, cibles baoulé publiées dans SMOL.

**Pour essayer :** exécuter la cellule de préparation, puis la dernière cellule « Essayer MandiaCompact ». Les champs français/mode/contexte permettent un nouvel essai en cliquant sur le bouton d’exécution de cette cellule. Ne pas relancer tout l’entraînement pour un simple essai.

Les métriques et le journal proviennent du run réellement exécuté dans Colab. Cette copie complète est enregistrée directement dans Drive car l’enregistrement automatique de Colab a échoué. Aucun poids lourd n’est enregistré sur la machine locale.


In [ ]:
from google.colab import drive
import pathlib,sys,subprocess,json,base64,gzip,hashlib
# Storage access was explicitly authorized; do not train if Drive is unavailable.
drive.mount('/content/drive')
ROOT=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape6')
ROOT.mkdir(parents=True,exist_ok=True)
ASSETS={'code/mandia_model.py': 'H4sIAAAAAAAC/61Z3XLrthG+11Ng2MkMacO05KSnqRplkp4mueg5mTRpc6PRcCASklGRIE2AspxM3id+Dr9Yd/FHUj+nJ8nxhUUCu4vF7rc/AKMoestkIdjrumpYrsl+SuakaWtd66eGk5x1ipXk5TlvhSZN3bXE0lN4EYUiQgotWCnUy7MirHx5ZroWLa+41Onk76146LgieS0l/s7J929/+LZuK0q+r7/7ipIfHsU3b/5DyTf/+pJwjXT7uuy0qKVbGTSANTVPJ28YPFZrIZlQMA0qNjUsyglXmnSSk/unptb3L78qGDo0L8+tQCVABKjKlCGRdbfnrNMvz6R4ea5qqVuQkE6iKJps2roiBdMsL5lSoKyomrrV/RAlTBUi1xM3UTF975913eb3VoR59MxSDgZTKf34ppM57hJMC6p9PZl8EZaZmP/kbV3w8nUtN2I7nxD429c5W2dK/MTnYHRNFuST6V9fmalCVH7s408/MUMle+Kt8qOfmrF7zoowZDl3+2w0emdGNxuZDWT+5ZXlB+9oftB++M+zOz+8z3a8lbwMU2aCaQ0OgF1mfM/bp6C23Q5rBZMgTOkWBqP7p3UrisjMacEzXq15UQi5Bd3WdV0Czb/bjtv98g3JMnC/zhB+WRYrXm4SayezMjgQrIyjKWyEfGQfzVbJYkGmZynttKP1ljlLHgfJt7cD3gSY7y7Ld1sGM5Cf/X4piYKZ8MUYs5blU/TLWRkDY5PPF2RGIBjtzJG1zezEgclFXSxlCrjqSu5sZQ05sCFFLA0MqbqGt3GSBqKkn8JFH7nY3qNTQfJ3rGUV10Bv0V5LrmIUl4S1NnX7yNrCLXUYLpRjrC9coLTqodXxId2UNdOwfFM/xndJWnEm45sZJTvOG5C8QEQk5JrM+M2rXrOW666VpOcnV1Z+kuoaRgvMbGaw38JwJlit1qx9+vBGa/lWKLBTtu42G6CMBPh00/IHAMAsnZJbEs+m8EeuroizJZhWbnk8NUtRArZwW7s1BoYMB9GOQqVefM1KxS/anJJ6s1FcD/SFQBIIHBXs75azlJ7j+pBi9olv7mC9gu9FzhdgMvMAA2i6hWU3un18128bpBnvhpWWc0q+BYCsvBe8CZY4Ssl8FXjzGjKvgphZWDEQAioeeZK6CSAaT/TrU7IG/sMyTVNKpvP53Yr6txm+HWPH7kNBSt7FMbuCJW/WVyAflsKf6zUOJcYbi5sZesOEH9rGo+ebtgYcFF/6sHwfHOWb7W/Akck5dJyuaD+FKRw2DTI9JT72hPgGJLe3gWIs/sGG9RshOWtjRx3YwKKCqSHYAuPuMqNf/Wqk5Ttk7T+grLrTv3dL0Gwgfl1GGC34jkgD/Gi+sIi2MbSACM5Zfs/dKr2z10zn95SUXG41/GYI11Tds4YHEnSI9Ux8SNK94I+x5XJMA1CMNUw1xLOC4OPxjA7CcucF7t4hMADm/WTuvcz9h5P5QHtV0RPxQ8hidDC6C6N9amMKfd7XY7GxXiFCQTeoTRLqnTBgMVTL6SrkvBERLLUP2TKHRBx7erpLqMkKmCVP5mcrug/zQaLkj5nVakFwF/sE9TQ4IRxgYrQM1H8ib7ibbKH3BRIZ+jjyN5JzjdurGujFTQNsumnsOtkakjA0vM3Lr4ZLdQJ+/3n7o+3rf/jHd1+mPTawK9tBrWo4wzYL6lXJ2Z734Fe3t2Nn4rZmAySghP0fkVAxtaP+CLIgNpCsf458agaxFbIoI59DWh857EEcFzeHx6NClkBlsoXp2ggde/1EyK4viSeCjLqDQuZ3hHYV5LMF6BTmnmDw69Q0KRAKtc7geFN0uc5CVweY37V031IYkRnKWRjzFHgO6oBhMU2nVKjMmmthf5LRCk9HAYbdpBbbru6wop7G6s3spKnyiTR+AngH3IaaB8/Fa7MyHF32H77kuc7XVrVBLzymAuP992ym978X8zyKtJyo/6w44Txal26xyqvF/xW8Bctk7+qWf+It9DVOTnKOG2X/Dt5LVe+Syn+4knU+VaMbsAhcyOrDVHyahn0+7FJEmd1hZ6NtmlD3NEvoABZDuCr+0HGZ8zNJmnYuA8/G0REAEHvmi6pvrW6uSxTbqhZFfLg69vSwF70eO3I4lRynAVF2EF9XSHy2Rnj9IFfN6U08NsF8ZSw7iBZIhraMjPjm09X7BXeal+Cd+GxNcnFvr3E+fLg7O58Hr7uluBxzXfP7eYv6UZ5we67fEDqDzQ6NjOJj5+mwU4iV5Mppjs/eun8vaziGvJ9xKRGy4If3s3EoLi6h+gsKqK6DWwlStyQ+nnbXF6bmxmbJ61nyEVId30NgqT7O4sCAdz7bBxYhqno9DLLMJUg0ZpF1W81o/3yH3bi71fCJj56MjGVU4gBcJ+cyhOQZLY5rGZKN5QEcQJ7Dfpj+w/mz4KVmlPTBblWPezsAOKhNZe6EbgT12uE2D9dGztGY17sXdmeQdoRRr7FHoL3yffP2w0c4UFvwjYfD7Z+Nwa/8q8VhuAml5x29xoBxtdKq+0YoHS9tIGGUiAR9RATexdlODiXZS9NklZxi7wzcTg/jrkc1iRKIxteYFlXn09FgRxczEmua0h09jS37adyJhIYAe2PXGOC+rO7wXmRhHPq8cZkFVZEk5bJQj0Lfx3EEWMMi4C7G8FISAGMylh9KjoSY+iRT1MoYi5VZHJYEpBbYm97d4oV5am727q4G5rYb+QIBJ3Jgua+LHlsoM66GkHNaCyUksEA1c9M0DralQ8jQ0Mwdq32sspXjNhnUvhTVQjbQdgtz5YKxot4zvEF3d58WJPhmBiq1D4rUX7aPNG6ZAAz9yMqOf9W2dRtHry0Zx+8ZcF5RL89kDmQ5fjXJRUtKBqpXEL6c1B2+bV+e5ctza06FaTTIGYOKr3zaCeiNg66QZZerEfQEJSbgEHNcdhVvsZYNAvHI7Ic+t5n5+EB91LjVXUPomhYzthSrC8lu1BwpDBPAsu30xhkxxDJkvDBR1luhlWm7ShuXBzreusNDEroqE+ljFf3wULK/hbYr0F7F40bKEgSUhcDJwIdSH3/a8F1EV8VNitYuY5vLmhDzw2jv0VtwlbeiMSXvvMyfsfBuxDaa2w9dsQdjQqNeZjQfL+LUTOhJSois+6P5cm1LPqq5Dmra2RWN/CfEnwwoo3kEGbmoKwBkTRrQDVAveUGsH1T0y2Tyhe28hdzwFhvaDEIXOtQJ7nPLpUEgghVDGg5pKGUjAHe8VjBM4Rh+yNAhut5xqRavPqFE86pBvq7lGPbOOoY/5XtWxt6SeLjH0gJRZ8WP7e2O/xOPOqspdBYKwnVpFVnR4W05dNdbf3vg7g4mPTR9dgFRZjXAp80r5tuHIfQ7LnzyAdrlytwlxnZBS3dU8Soh47EhqN3PIP3cDGQM02dpL8msisspvZmt/BeJQGNkApEAdCB9ytotrAc2wgAY2BvvQqY2FpDWWqXqSi1kXQEwsFGuNxp5Uc7tgNUf5Ohs0MQEe/hkYDQZHTydbguHiHGCWrec7Ua15npGPiO/2V5jqRfcOcbH0gherY7wYLs9NcqAAwD4P+f9a/zyN4jrYI+JCRDDna2fQGDsMtJ8cpRatE8tVzrlpfmab9oTl2vcuV36zeCY+a4Zu5lRfbatqu4aqOHWzUsztLJAGF7FJpP/ATYgikqRIAAA', 'code/mandia_seq2seq.py': 'H4sIAAAAAAAC/7VZ25Lbthm+11MgO9MhaMPcley4ibbMOHVs98LOTLNtbzQaDkRCEkYkyAVIrZRO3yWX2efQi/XHiSdpa6dNd8a7JPDjP37/AfTV1dUnKjJO35ZFRdMa3c3u0BzxospPjwUTNa15KZClIYgLXnOac2WXKRDRuuSSRZN3Ii0z1ki04hkspJpAsByxGtE0Pf2q0OkXlFOkykamDGVUKJRu6X3D0CovU5Q1KDs9Wh7R5C/HleQZA1VeobQU+zJvNEM4QhtFc6bQczRDH/76PbpF8vS4Pj1KJlJN/1qvRpOPQJKCTaWiolYgVdSoalY5Pz0qOJOyumaaYMUF5bCLRMAUkFCFGsGQKJs9o019ekSVhOfTI5h4dXU1WcuyAOVrmuZUKRACripl3S0RRFXG03riNgpab/1zXcp0a1mYR39YiN5iJIRfXzfC+JHmwBS9t0SFiUVSgKdyT/jTp7sfS1kQ9FNZU3kk6AOtWfbW+OotuI+guwf+4ePfJ5M3raIT81sHHCjWfDOfIPjZlyldJYr/zAAF4LMYvbr59rXZynjh115+88osbRnNlF+0VLt9MlidmVVm0CGTnB6ZHO1l7NKe5bZei6Qn94+vv7FaUskhrnOkagnLwdbAJTB7Gi7JjkkAnz/2td+o2aFuhX/tzJJlVTawvM5Lqjei6cT55nsAidABwEJEn8qsyVk4dzqvUZLodEgSrFi+Jul64/b0j2oqJnEYtTRhtwXU0ZaYP7u9e8hiOB8Zx2lOkfeieQEHXF+3+0NG9zGo9pELRiV2tP4MWXGq4vc0V2wkfXfhkNfnmdXn6cP7/+UwOPq/URhCxGKLbWxFhG0YduyobAgOvQCsiCBJfIjUllasXZWsbqRAdVPlDBu84UMY7Tl7wPrAKCZhVEsoU1BDGJ6SWYjWpUTmFEAIWUV2lnYfdgoB1QOVmdOJ2IIX/1gKRvZQOzP7qKC0utVyvVasjm9IStMtsw4gttA5b3zOrvvYguHcmqdsaY9CxQPR7Stf+wrNFVTBGmkNO/Emw8k+NtobYvvQ0SIGCjuYmcgYbp08vX2BX0t/CG/vyS5uw47vnX9C0q3t/NqA0SV1hqJagw3d4mZp/bh4MVveajVsAU5pjbGnILuQvJiF5GxruiR7s9VKKKjamYj2vWlCPtDIkJnlxZwYAJhf82V7rFEscfG3fxDUfKv4MFSjXfMgvpsOjTbyrPoUMLAB90W6wGNtVsb2HNB5iOxDuHC6oD/FaHDGOtz5/bk4P2ht6Yw4xu8jlUKvzpKsrBNoo1mT1gltayqEOZKsYrSGIlkzmTO6ZzarttfXbS5OQ7L/QjrgLRJjrv5FuPJe7BwajmuBL0v4OMqQSPcLvmnKRuFeVr2YhiHBgMB9aCMAKWsxr80Pfeuw45D8M4w2u9+he9waNQX0+Gnsmr2vnr292dnesIxq/8RdU9NC7WFos7GdEHqLujHqYv2DbZCWp31+stgZWPcsOUKxakXjzghIc1cM7YnbQ3x43krFx7Mw9Xe9yh2/GfALW9//wH6r7wkXGTt8QQR2QBgHesQITPTBJW4WiWM/hZgsNAz/8PKreGbBEWzuaTCMhnEE6WzoHl+exZF8dmHIu+CHeDQDmsiaGtka4i0xGg5xMWSXylKp/y9ybJ+woPhcc+wHCk4bamJ0tAdxW1XDQVPo9Se9NGh6Y6fMj6TlHXufjhDcCXfV0eg46nVfwqfd7HOL+0xj89tPBX+TTU/Mk7lz7DulC+Qwb/yA0g9A3A9Dx6OvytPiR7l8nq0vbbaOM5zgS8Fss9reQeG68ruVU6DSY/cQ6qxYsSzjYqMB/M6/GAh3dyNyOevcLSduFfzIVY0Xg15gslAPkokeIm1v1dyGN6RwOWLtLklj1oNSZwqZZc6HzIdXrDPmIFvHhXhJ+iX+jRWHVlXuhnPj5m5b6yNowUillbIogNcsqWC0KBh0dGiw8/Ekp0lAsUw98HqLcQC1I3pgfLOtAxJk5YPwb2E4B6dokQZcNE9wBcjN4uhmdq1v35G6lzWePbvg5ucXnOOQ+Uajj6eg37bMOphpObjoqQuqcsUFUIuU4YLg9npD+gDSL7oKT7ML+hbOFq92VyStuhbULkErmsU3PQVswtrNr2LY7aumh05DEFFxxDC7RDTPwduScqiC/6B5w95JWUoc3Nmpf88zFgX95B5mxdk8r6OrP+Ds2uA6F88PsVnHfiq4OHk56LXzQGu5jYq1HO59GyiE5y1CfVGPAD+4sdUyckP/dPmdLwOR/zZw7pe3dod1X6f0E0zb6vT4Hx1lZYW3+sq7WA78xUnrMSaagkkwBffTfJQO0B1bXw6cYBqa73DqQotTC74cNietj05WSC2sOvXzcsNruHxHub2a90uBb3WdcQ6vZyHFlg3RjX40Hdudy93f2eQ81tlesKKUR9eSetjyILztI8kBxmGlfzTsgUqlkldmhtGHerIcq3/q7r/mm2BuP+NhD5GQBF29CuaqKXAV6eBBOpmgdtWtX9fgmK8v0PeD+WJlpgybNu0JR7Ikgf/C+rP5whrMAyjiWVkQyGRUgY4AUMEyZL7+Bf+aTN7YSxoXa2a+gJrvgjicaGs3TFhoGWrv5VWpCIN/BT0kdbljQsXfvnaOMIQRA69Bj4SLXSwA+vb40Cp365ucV0AnJYf2FM7daGF1hOFRQUYt7OLS3yDhjw3QINpOk2G4ezl/a1QfcV6AbcsB41u40T1A23QJqL1us6Hrj50femhwOHYCrTIeYJp2gC9H5vNsOJztdS2J4eLqcmNxA1fIZWQ+NsJMAmgFBcCjTlOfmuZYOPqQAIxiCN18JRndtXsXPWGol+dOdjC3wpziE4MW85ysjrBgv3B4f5zVl7ljcuO3HQZV4sT74z49zatPlmfuleVM/weDmafwQDmdW31tDjbBDiZdLOvJvwGa92HWuRgAAA==', 'code/mandia_stage4.py': 'H4sIAAAAAAAC/61c3XLbyJW+11NgnMoCGDUhkiNrx6ThrGzTjnY9HpekmaSKYaEgoklhBAIYNCBL8aoq+w652cpN9mrjvdzKG+hqqvIU+yR7fhpAgz/yOLuTzJBEd5/uPn1+vnNOQ48ePZrc5Pcfi1imc2mdDc+seZaWxf1fk/uP0hpZaVZdy7C6sfIsjpSwVvf/tZLKirI0hQ7q4KKKlrJU0FwV1tAKi/llXMp5WRVSeXvnWVVC7wT+VSEQWoZFBN8dTew4Cle/Edbp29cCpl3lpawK17qGBVhRXAAVuZL4PUyV9bKIr6W39+jRo714lWdFaf2gslTkYXmZxBfiMlT0WYRplK1EGa+kWEGbWM5FlcbzLJJRWIaikGKeJQnQjrNUCdysvClh5N6iyFYW9pknoVKwSj1NqKJ4XtaTlhnskPvSVy9N646LKiWqYQJjrFfcaQXricNAyR+H8G/dFfj8IksX8VJ8Q+3wWyxlKouwlEKV8N/g4hY4t3f67bfnvt6j9w4+HfuA1pyWBxFy5OCb25f8SYQOzho295IwXVZS9bjlBfA3nJe9YX941Bv0e4OvgFASXvSKKkVuHfBKe3PdTw1VT5ZhLg9td+/sfPLuTPzm+PSb796J58fnL34t3pz6R/1+Xwzh38GR+Er2DvfO3k1enBy/8af20zyMntnCfnqRKfqU+rOEA1IJbJB+FTJPbulbJNMsVvx0FaqrZ/Zs7+Xk+5MXE5/5HMnreC4de15FoW3FC81+/OnFKgivwxh2k0jHtWSipGXP8wpWHsmFpS5DJ3dHhQSpTC0tKR48HT4+cjrMzV2vkGHEzHdc17uUN1G8lKp0mNQ8TDOQpjBxVEPQtmzvhyxOHUPOvDQrVmES/x5W/PbVC1so15uHSi6yJHJwkjwJcTf/84d/t8Uj+5HrqTyJYRaeJiyzVTwPUMKdXCBBd7RnwT+5v7becbmCZ977uLwMVLVYxDcOrAbGeXlYlDa1e+8L0MkABd2hpqha5cohfZCpAlUNQjWPY/9VCJwTcQqHUfpDlwfXa807SyPu/z1ry8tmZXSAKryWvBTovXVGraNOw3AeGFZlBiwttUiIqLzNpRaWi0WSheXgyEVBYTHyqNnX8kMi0iq/l1ZJ0szDQnObzh29s+1ERoYEYu/LAkQDDtzdIwI5bCMsZE1Dm4wriX2Kxrqc10/ECkQnUQIUJAZDoASMD9r+IpIoW4UiYkX2XvlTOkvYaKScxLUWWWElVpxaDhqNAxsmD8HQJT29EJKKxNYSzjtlqUtgQpD2GZMG5sfyk9R1t88knhcZOgcZ+R9aTSqmVzOeoMAJcG/I8WJqEwF79oVvE1ds6nSFnaY2kAKHAeYC7Ml7WdizO2Yz9KBh3IvHCfsadBGEDKwz/ADdBqp8LPWYrLyUxWeM0WJBUz3zafQIBShOK9nphMQXsUyiHcvuksR/wi5zaPDDDPJ9+nK3Qeri7yBFe9kkhT4R5DXNwCP+w4VwqL+gzoLountGN5A8h7qmt876ScO8jRw8eKLmQlnceI5Vlmb+dDZ+H96Eic/yiBbkgB70LsKsSmQPhVCLZa3E1O7Jm1iVqtbL+oxQTnEm7rNLjLunVfimkkAPd+PofxCoh8sizC+RukyrFXl5B3SGTX5h/y7d/1fnV0/9qffFr2bu79S+LeBAcHbggbspIKXfuJyGeOs/1nvDvgfDp34iU6d0n/pP+n04lcjwZKVLp2oey+acNd+9MM9lGjkf7DiyR7BM+JjtL+zehx/u4Oj0GY7K5hTxawkO3R7Z2sdDN4BoqGMjG0kC45ZVmMBjlsFRo4EKQOVcBlE2rxAINvPddWQNaQh7coOjCklgk1DGvIhzQnnWb45/e/zGAogAYHIuy1JasgW9HovHL6zJzTypFAywEKMmGaEnq7j/uICu2FNZl+C+AdJawARgfKwUUrz/s5XYcVoSQUDBkV2BKOWXBfh7j2jjMH+ak0DkXfmH08GTyd1n/vDxzBDvohV+fIAda4UiEqZWrWkMteOc2uqSF0HfrHWl8SqkHo12NN1aDYFHfuOhPHRhwSIGjKXKwmm6a5lDl9rKTWekw77Ne/5u4iCqyK68jnfzu77Oew7w6428lokTRlEAjQAdAgWwVDJEYRLaI/q1ZzSGtVpQFn7tUb3nuTzn7841QIeLQMF0/mH/yZFYxWmwKOSPFZzzrT8EDBSXcZgEYZJfhhey3LlCr+7huELlco6DqJ/yNRo21gIarXyygl0TvK/N23590qY9/jm+zzX57tGAgE4LcB9YmwxOC+euwQWwhFlI4GvbYWrVatmMPTxgDn61f/rT3/7N+umPf/uD9dN/W/f/gRoAQNZ6BVp3/5+gFvDrj9ZzNMP3HwETQKznwj4+axStYllkVa78D2pkxGywoEVYJSVGZWB0VclCrz7pvO/GvP4IfYdEZcfvJuUXGcRCIB1uAyW2H1Er5qqY+1PiOTIQTj6II8d+au+jAUerN9u3n9nubN9gX9Ec4Ix4sz8dztrzCwuIp/1u9+aYN7pr8wHLcJ8NjvoWOjI09EQFHj0Zdk15ve0pUNVGGFZ4AGOW5SV89QfjDRQDMrTyP2hjbI9gLrToOAEYd/oUNrAIrPPdmE9s2srvzJxpVnsPJNnKrD6Wra6lcQotydZ9GKQbZ8HKZ49qttRr7TRo/ty1B60+Ey9qOAcmoLiNwltyVovyPUBh+LrMwBSgQ0vAkoKJhsFaoTR/1GwazQBDiS5q0q16FbNpxz3OyG1vdKEQ2maJCKsoLuGoUGDtEajNh4i2i+xWrCaRoB+4+kh59N1x75gFIqLnPEXTJuxaaOwR6Vz9E1lbW8SAo2mQDoi1W2MizB0EtQTZr7NsmUjtljMIDNFsWrQl6MqgWpC5hBgtqVbgwgHGIZPTLNA+uLXGQT3QHp0XlWwkYRXe2CNQikYC+MGToQDJAjgRl7cBQxIYG8QqAP8a1PkJPHkix0jYDMjZib48Pj/uHX/38uScvagg5vNx6i60jx6vU7UhkhGN2xrGGVF54e4DILRb66O1QxvmvACk4dDsweT8+N3k0BbGaF6EWACQufRx+a4O6nTgfCX4dDlEvQhLCOLpoAXnV+osVFb4b7NUCg3ZtNtlFVAQHKLdRcukj9SE6kRuNi5Lf0pGkAzFbL29Bh16gq5ykRgqNdpA0zGZ4zBdSmcgSMPd3mALQgbClAFwOETHpKADDrrdXPPNdZ96w8cjNY1n/hFvj7akEwkgXplyHJpKuQIkyFmFOf4W8Nt1O1kHhIyajz5/MPcbm75GsSy7FOH3z6SYAOJIlO60qJIkSOIrqc2a6A36/dayxWhnSrcbffw+zmFDNKPhzGjn01iw8XRneoIS7CZACLW2kjFPV/cv1/tPB+D2pqPeYLY+kJe/c2C5bctahHmNQu+UCQlVrTrHsm8+wD1yHukyy5TU4p6KZZNPmtKjKSpW7M5aQWuFB5taQyqcVHSECXA2TpBkSjHUFduXucogXgZj4/c115NsGQMkpDFOZ0xn06+8OQiNCsBoFVl+6/Awj0Uccy5gdXPp9EAnuAWBLfwEaeKpjS6uiJdpBmgW03w3PsqKKGRUEQTybeCczYOCdrnNNxgMnHCY6Bc0GOJjYDaweO+fmF9xupAULgW4LZ1Nk+BQKxQ8Zg8bIX3MmhfU4mFHxx2vZFnEc4B+bVaHPf1nul0DF5dhIuYI8Py+J/pdewM8L1vT0ifToj0tTztzxeBozdCA+ohbEfhsRbvdp0RyRP/dHxzVGtCNzjExamQ3r0XqmyKE5LsDaBP7Pp/7tTum7ez7adNJs61eBUABJIiZShh4QN2BY1Uui2swulGDiXQL44Ytm78zpRGgoNSQOVBzECXw+iACN1OejBXohgJXXo6Hh0/59IPDGrzBPLr1jn0ROfE18aAaUFbcCgUyG2JNqXZTqpQQGFBZQp9K09lbXcF3zIyAuihygoIC2gBD0sYlZnkZryj4ZLGl3x5VpHgVmKYOYY0Q7oHrSAr/zamAOC+ECM57IrwnX7vivYyXlyWAiHl463v9AVNuDQNTfl3/BlVdhSkhISkj53BYG3NVAmVV+s0eDuz5pZxf5YANyh62exjubW/GkdjMzgv4AuoMot6espBJmOPnJTABuYkj6OD8vqD/eWI6EyxVNqiv7dbemabezJdh2Bg1LiqMHNoCGNwggbiaDQmWXzSDVIDojXk/1lCXSAB+qA+Wkpj1j1ae6SBwioCrYgRA9VhqBOgxbo5yV8+mA/ZujsdTgAmpc91Phas8AUBZAJbsxNRUqIDe0NAdQU3r/QnWoJjuLheYtAJMlmp62LpOjkuaOKYzdX5bXoIGrnX+TAHQxHRCQqcXkAZYUaRSW4htVkPY72HlIMtgwiL8qcnDN5xA2wZjbXBcfO5jjWJPJ+9OT84mwdnwzJBuLwW1Iw3fwLFclBmTWUUJjFfSg2Utgnkncv/0AYA/hCPIZXgVrOQKJqWjUHo81X+oLgUY2JR7nj4Pb1HQMCZuxHfUmih7TqVde8SVY21N5gvwnszaEZJtohIyZTAeP4QW6hGPMSQZxrZSPGolvtvFFOBRK+jLRtApZqtFVjNk2RFs6NEIocmx5YbIujv4zMU1sCimiI60FC9rKXa7AjYimd3mmww57grcSIv1/hYx6GkRaYVyVAu/KZyjRhMwdS3nsaK4z67rh1ZdZBTWq3dfDS1tzw70xYU0sxTgDFlYKRh0CJBhA2CZoyrBQ7KH/X4Pj9UCfLKqcqu8lCn4e4WVhjKzBv1fWm9Ox1Z+GSrZtx73+1YTjlosFW0ZplN2JXOrpRAsqhGbGi4iL7IliLnS4emHq5EesV3bO9L4dyi/wdWW6XWmHitlQNj3+yPSKzM8aXEXqTwtwNA51gRGB6apxzgqWBbgfBQldyB6T6VhKthygi/ynXh/4B7w5QVcSfxUfyc59Qb73pMvvcdfOoN9vDDiwQE59CWPv3TiHvd1D2h1qv5p1Fko445QgVAL7qZdJD0PGM6M2k6AlMAVAaD4klZoOo74KcjBGs7MJSDhqdMtlgyOxMBz20xgN/3OFY1v8LKOpe4/Ul5bgnhZZVyES6ksWeKvVah+xCIH3d7BwkkkqxvrOiziEPRIeR16Grj5n4DfszF39HX/KTAR9uT+8qvZ1m1xN9yP9xhCqwa8i6+FN8QHnOESh8Ib4M8OF6hPSze7+AGB6TUgG6+bruNJUo1IKL+A869nR8l+Kl+HiuvJNg2HMXhs8gdbQoJUFwA4MKiJbmRYhOaT39nSwyECAmmhya/HCsLrP3bXC9RYM9JxmVpgVUM6RASC3xD02XqFhg6mfodYclIUgFLtdwDQJIxMLRwhvbU1OczDL4nOAS/G9S7C+RUIQoRKWp/Cvq+76ogFB3gRYGhgCwQDPHSMpgaCGP2rNTX7Pj9qJsc7NX5968qryhgC23kS52QGAmwNtkH3gSckbiyIF2gkmAsajZqOVOZOB8OR3fglhLhgtjCxHgNChU4jDV4AtAQYgGwgFxgm7ANbaI9OQZEASUojp+GMONxM0nWmHT6maTHlGz/z0TfgCvYHvIZ1ub32twbYUzOP3cSgY4Z+8nq6Ech1NVRb8TYvz+iF9sc1pmY/AEmarZmTjuQ1GAY4nhiCMToie8TCgN/rBHz9j8HZ74/fnLw8Pj/59u1W/hopT3m9nZW138GNPW38/GausAXD9N9xx9Nio9iF8wx2PAD5HkB0dfSMH3cPy1578uRAsdlExE2+xO7Cyf8bxAriFFP9mB6ny4ZBnP1M1EXIeplXfJOuAyRX4U2NuWEqjBchFn4YS1IqmhFGK1tbINw61rszwccz5h+plImTSJJ7daqCwJLbxrw1Y42LHp8cbFwe0YdEatYNmUmyfl7IvCMK1kSNQLhO/2gwhuI589t+/GCjnvHwdoT+5Y4xfGvspajJGrm7T0EynT/SBB9OGNKJBtpXAne0XePkLdYxtDdFadL+8MmRedVOhfNCXiSy8vR89Y27F78+fcUZoKrMq9KfzrrVXpqiNRNrHhjvLQd1+aC2tkYhRAzE0FhUq9Q820aNEwuns7rUyWXhYFGYTXWtGJW5ZtTFPDa71PVhMLV6YRF3Mer3WIQXtsxAGYe0TdiIWUjVpNp6Klg5VHmtcFj9oyKQtlMrH/novM+KKIB/ZeEPa8eyAghd5JViBXWwCtRdllEMYq7MxBR7dfe32WvmdtOQbbqSO2Doelm8CvJc21RP2wWAYHMs/wEc41xl5/LM2uJc3+80r63KXV+We9BZQZoFxGScBsEXUMDfG7tBz0CCGZjOpSycFYfr9UOwIXeChzysMRfw4HIVFldaKDXufCDJ3ql01UUbPayV6NnWkhar3NHhGN2AwhsVdE0nSVRHoSQ4AiOzfrSZTCGXsTOTs0UD2XcIkktf74dvSvCK3Y10DS/M3+mwxmRVFVU8xzdr/JgOtnPg/7KFjbpmtliAyWz5pI3HJljh+o7QC+bdawW/ESZndBfBlAU7khufx09HojeYeRDxY/mxNxBXUuZRvFpDUHp72uHv5p8JXPjERyQVtbHTBJqTaBr0by1SNAR4XjpGT/7d1Xvg2NxAL/yJ6hRB2BowO9osEdGdDmf6toIKcryuQK312IO2jybCElW0VOoFUac5BDD6DY0gXAAngqNDVN7mvQ2Hue9u3Erhnx4V40CyEG5hasgeDdC8I+ug0+P1ZNTgyK4nBYuWS7pLSQtEaLHYp+qg9S8H34v6go/S1wYKAPFRJNHpomzYutiyomzKqHvVa9frFAiYuq24ogBAI/pTBG8mnU/dPq+RlU60kl5tvuEx6rcvCIA1TKtVUF4irlLOYWOBdJTjNzftqQG33X0TIaBLi0HgelwUYvMEqFYWFHEAasZB9uhDThHGiF8Yae9SYqu3TLILx/7Sy29tvBSDF453j5g+yAXx0B16sfWSpvgZ15xnsCysLNmjw6GoAwF+daeN9EB2qpRBpoxALEJwSAtMTcLiq6bK2sJs68WECttSjTHfiXehYGpQZ6slKWx9XzL+vQ77bE73Wql8Ty+VaVQ7tq77Vlu7wnckQGiwIgBMozePyhiErvMaGaVZ9Xi+2AvOr4Bd7rqZM/ntu8npyTeTt/XNHOOoa0uCkgvHDhHKRpKbxTDAY3X6db687hUE17JAvQwCk9d8vxXE4Y6COtLQU9ga7qO51+RZLwlpH+TxNWZl2ptGysohgLVQwiV3B/ZS2gvYgu96KUGJHEYqiQUmCgx5JelEGKcZJ+bZbdFcp/LEYsGVc8e+vL0oAG2Kfzz6GrNpYYn0cUMNzoG24deuUTvXYbnv+xZVamqi+MTeGnszs9aKneNOENg0US0Bm5s6l1EbJYVeLP3mhTnz5i6iWzyx9hEE1Xpxfr1RdEr1s/YhvyLWbN5GBgXoAeHT7eae/eYVPQcDeq/MnDqR0kROfLFaT2PUu9S88dHNdXQEDDW489deRjPSGh8AtYKnivniuyHCG8kGXNWdQEcZXMlbrni7NSZae4etvRZNQuZ/VtXdvMmsyl0pJ74tuTGkDeXwVscYQ38CtQ1e/P+558GJ77P7j9o2gZ2/kXSh/6shmptVjC9fnbzEl1JpML9iCkD8/iNoGV8M/LECE1TdWEOUgPu/JOuZ8CZPrNgBdrfepInhNPwkXF1EoXUzuunEeu509NWwm23jaAChvr8j+G0yyW3825Szka/NtQ8mZfB434cvhlyuMkxHrx6Ya9c9UzP2Hh66ho3ILuTGWXLsKjg+1QboDKSmtFiprC/gFI/TrLrMbLRGz7P0B/RC938GExJ32ybgeObSwrMpK1BpZV3EAGx+Bb1WsfW+SgFBWld5SJ3PK5BRi/DJg/3+GWQjixUWcMq4EHDqYHevs7iA3rcw0U9/st6Dia2o83HdiNUTWCLSLe//Wm70XZNH5o2RRtX3YKeHnXvhzKzmjre+VM2ZApto9Ox9jBDphRGi6brtKzea1817N/xpvGtDJxggJOF3UwKiYd/drZ2iFkT6/oCE8Ap2ygPECf56SLqmJ616z6b9TYvxC/CX4TLNVBnPsVgV3X9EBoZYyYJzWtTvDeiX0uEMI0xVWmFe3P9F4QpWVckYBvWfuW5oMiV6/AfWhCo67t7yMu/KroV320NzYQSuSEFG2255jd8XeNMz6LR4RQaecSAAhHRugkUS2YBz+Ogk2CbYxkVvEHSeKuDbX5xw18/cgxTTs8gb6KDDE7MfLYV64dNgGeadlp5Jp8Y6xkFh3pLRigYmSoKzxxYQI/QsKdakjcp2m7o2nKQ9wh+6zEBXu7XH4tv4iBngEaZQ7VYwAzrDkWERWwIBSz+ZPiSOFnCHRlC01+oBMmuBb3MjNBrhj/bFtOYkkGT9faNov55l3ZFf3ZWTRcRt7Km+bt4Y950jaXs92mt9CaA5sTMJNrG5cc+2Q6PMEJ1vvZ2yvrrJOJ9Tbuw6hL2Lc9BoGJVOQcVkA5fqSlzWFkiDEUyd7N5ePtn+9sHW8MmsNDH6n9agfTbVuK32m2q8GVPAPnGQ/osImptMyN3bUr46nZx99+b8+LwFyya0uxpd8w0McW1ek9RvXCBivarfeUbZv9te4OpM9+7NyeRs+2yGEv2cg9vyJwG2z4/1AOLheDn39GtUTgflSxCq24ASF3WSQFz46/xvrofWdwPDabtkQh5tDWvm+xcPtO6gsFny2qSzpc+eKS0K4kFuCfhvnmAkGC9iCAFnxJg99gQpvl+3AlC9vk0TDl6NNoTwatZdz/Zq18ztLGqOKCritFNg1np9XkincxKvYvaHuP+pnaVonPnWEmxO53TZBtCD1+++o6wcvXfTmHMs7QGY65iF18fnJ+ffnlmv0PVdWhX++RQwI5N0mcTq0qKAF/E6rgBC260hLrQ/5+C/iuLsgN7+0cG20rgcuW1dVPwWOr72oGs6mD9YxSqRIQGAX7+yLuZx74KoNS8vgdRXF7gcoAGqAyuB7dB24f/XMilvCRingLCo3DHbkV74pCnAZJPqvO6LA71CJ5BsUvAc82j0Li0n2Op3e+nedE5/IkO1F6mndhDkt5wBDLAGUQ/h9BMgyEghCMG/r0F/XIM6cCNYkrPj776fvD4+fTnpfT85PXl1MpnwumsRn0twu/TiGJwTVSFwyWBiaS+AVD/YmFHjJvwDHQkXhyAWpo7QUxd6c7LR+NZ8SRE5yJZhm3P9qhlxRQdPNIULcLfWpwCkfUUZ5ojLzvV7XbukfcTCvisftHP3Qm+882bVN8dvX54cB9qqfnt6HjyfvD5523nLik97q6HcMn7y9uVmlmRztrPj7ydb5+I1io2+W+nqJDmvcA8ELaBMVhD4PsgQRgIgQCPO/+79L/y4aqL+SQAA', 'code/mandia_stage5.py': 'H4sIAAAAAAAC/7Va/27bSJL+X0/BDTAgabdpS7GNhBouzjfxDAJkMkGSyx0gCARFtiSuKZLDJmVrDb/PznPkxe6r6uYPyUpudnH7hy2yu6u7uqq66qtqvnjx4qcir9O8ieq0yK04XWRf/5BW0lgbmWaZbCprUyRf/5FJ69Pkk7DioiobZeVFs5VRY62qoim//uGNri8urDJSwqqLO5mnf5fVeVTF67SWcd1U0krzeB3lq69/KGtqxWsZ35VFmtcKE27KTOIhiXJlvanSrfRG76Syqq9/lEWu8LRq0gRMYYjcRHki89pqcmnVVZQ0MbEtLAlWwZW0FHZDjGAs/tIoK1YNnrJ0UUnljV68eDFKN2VR0aB6jWbxN5CIdaT4pcL8xUbU6UaKDQaIVSxquSmXaSZFXJS70bIqNuC1juIsUsSdmS5SSRrX7eR1gc3rsZ1AuqGf25Z2NO0qjUJVRyt5iZksdalp2w75+wR/Lf2v3ErqwB/Ut0xXo4+//fY5UJce/XplVEFI57YmPyMRR3F9pibqTNZRKa/s0afPtx8+iXcfAyhOTOTZ1SiRS0vt8thJ5DaNpeuPrHRp6Rev3pUyCOy4SSLb58159OwRwboqaDeOy1PIPC4S6VTFvRLYOk3DRqKCxzv/0YbScpVFtbT92VzYpFqZ0POTtSwq6w6GYs1oVJrbwt5GWZqwZeKllqq2509T+RBnDaiC2XzERBUR0YJYy6qjaiXrAEt7LSszG2vey8qeu16aqNPZBIRMWUfqTsC+pSgh7rLm1Z0Bk+KlALnuBLlwWo7FpbDfpArbtRZR0eDQWL5lnw4Hz4kdGGRTxTKY0Srz0wFbetyQI5J3JnNHk7h/HV9fWGCSmvSu0PR64rf796KylHniPNopJIiV8QOR0p6gI/x/cqexPtySZoeEArJRpzK7DfSPgMVD3IEtt7LaJdHOFkQd0D+XCLX+ZlhBlVmKrc1n1DfvGdAc277+JR6IXeKCfoWNpcFhcf/kao3xPCRsPbU/UEffOuNRc5+OGax+r1EzIBx+E4bTSsLVtNSiFdPoP7S9pvlS4ljEMiRVGGvN5X2YFUo51JYJQ9qfAG72JOzQcadFUwePT6PecuDdNqpn2ON3x2UzLOooE3z2VXDhiQuzxZRtFZ5QOheCNMs0rhhfM5kViYXYiZCO8iKq47Xun6V+ejq+nrec0cj7tF7DUXikYvlQY9mtyIlusB+ejUczP6fBMiui2tm6U83ZaZCjE/vS8gwebSKmE47R53oMaw9tA2Zt3UPj6PepE31Lr5qN8zDTL3Pe9wPtGwt5ECUcsuO65zQhWlxtsqDCy9N3tQX/X5idFVlC0h2oyrAAASgEKRmuZC4r9h09SetZ5jO7kmW2w3npJxGTS3fE6/AgQ6VKGafLNOaZYFXEPzyDSNIKoa2odkKlqzyiINfOo2oJd6c9bFYF7z4KPlnB1YVQ0VaG+m0Mx4sIKKuqKeswqoP3Rc4b6Wb2Nnd4drQ3V8HnqpGw6lTVYXHHb9AsglAddBTndh9az6jLgyeaLuQ3x1AXjRlZRUlOk0SPp3Tj3STR5r+1DCieRBsJVqE32lFWiQUCiQoc77XwXr9yxb1MV+s6TGQc7QLvYgzOVma6X7QeispxPQSkJsoQz2TiXE7cKUkqqGSUBaopZbVNFdz6xVRmUclP3nSN7YJp+HreBqvWWLENG7Hd6SJSMktzyfLjmMUbZ0GZwxgbVkCWONSL2F7i3GulIqyVjW22oMIiz3ZaulPjeWI4vlbH9jwIuhdMrQVEE1P4rmXIHhYU3IEoMIU4j/WzlAkE0JiVp2StB1CfiggTVWGVr6hXM08j0NCP4uZ2jAYuNKjrL3f1usjNgJH1Z6L5cIkwyjKahnr6SUhjgjQmeo0Joy9hlCU6TYlOOZjpbj4M8TQRgjpNFRp3Iux+zr7tHnzAYuDmEno1a+CJVgl5GXoxC4UDyACxQAdWDLbLCtbu2B9vP3x8++k2vP188+H2yuaTKpZZo9bteRoiIJhnVOFUAAx64GsZxkVDBxbeyGK0hMPsJObU7lPGcKW9zfi9i7Bjxmu2r/GiOWDxcgU3yCLxacI2gobsSkBPP8LYlK9pBuYE2t6cfDK4/c6hPfkrb9XZGrvy1oqMJaz27AwjOgMYmsrqmam43zAwYHMlLQJ7A4P0jb2uWnt1903BZxM7Yg/+wOz2TcM3Vnh6RF9nrMneePzWUodG5Pdme9Sc/LbxibR96cGnbdI4ZKForxLDZLp2yiucgdcF2FohB1EeddgCaPgbR2JP9f/CCRlsaU6AC2phPxtckJl2J7KNZCbOcfQZ+Nh2HNiSmeQ0q520dcoD/MeWe7EH2HtxDaWipUXLiMfv2XM346FuaDY6dxcGSg4AFZ9mFhsfSD2tjuXaDeN4FOEKaaNDnq4uwhybbk++xWAhcAC03POXFyS29Ef8svl6k1Pv1Yl3deKMTykvBOoCxqKHMj1x0rOXF+75Jnpwxnp9enddA/nKFWMfrM9xNDSYt1wBIMH7I5qe8NKExBZ/I1FvJUJfCxgJconchCeaaR+ROOkQOOY6mLbgkeDhuiiUdNpp4Ge+jyEJnoncZFFH0KTwLq5ck6zkhcl0vVQt0xxrOEzv4njDRK2fKU6n+eoDYY3bqgIGsGFnZ3qsxfCQ53L07k6Y+tys7mIT8d19VCWkvlYyp4EZa6AsUXgJ0Ai2C1BpaKckB6Bb+pn2ZwYtup8ROjGe515Tp5ny4iwt2TZgFNUmPIZ8xp6QtIkwXZLl6F0YqKCdriwdE2nZjH7QSC/gNA4NgA5kHawvuT08ge15cb91vjCD0AAWMg07icDjt4/7B1BucVhM5NMRL/xy8+7tm5vPb397bwuajnyRlzSbUjly6+6HQt6H3M5scLVKFxlOvgWvzU3PfMKP3QHl3Vm9Izk6/F/2CCyD5z5hKPQBxj6UfPc2BN4++xMi1bM8H2BSCw0waXQyBJTM+p8DlMfRIs84QIx61HedVZdv7QME7fpZUGZSfpl/M849D4j/9ij8pHMs6Cdd7kJExGYjGaaXFRxT0SjyOm9uv7z96XbaPRl5a9flsHwhBcCmoCuBOVuIfxEqgKDg5QT52Sa4fCXWMkpUcC3utqF+nAhdf6nCLNrhXAdjOMS9holYLvOQ6K8vRVIVJeX9HlkHO9Fg1tc7Zi/FK/FaTOZ9yWM2vhDjMZp0QSvsw9OVOze4EQ6c/bbR4sx45wk8TDtGbp2NWHRDHp8dIX/iif5k+mQXwCV1taPzx369rVt6nyXVDaNq96ZFIwBrkbLqhM9qnQSmFup9wK9TJ6ZiQBIfZmzj14M8w7xvgq4a6RCKnZrcmfNmSpdFnZzbyybL2gqe6JRqEuVXJj2+HGbHl//vPGhL+ye52M/QNVP/xuV4/mjoXFrpnR9J7I95mMUhsebjT5LT8h0gjTqX1Jbg9NTyd2jD6XuBYsVi+OaKu24isTVTDfJd8kpUXJ0PC2Z6/DKVVNiBjym4qrvthhxhQQ+jtY9NDkZmPN1cS/XoFvYybZppP/XmeHdAMki7iWCYhQ/WiWZ7nnQeBIuDFp6bWHjmXfXgI+0EUlt/EMU1HQq2PfALj2h8abhI63uQhcyx9gxw0q2uW0lx9mZcs2pHcarX0gPdhXEWwY8nnX8B5oG33/m9Y269tvbqGwbb0OeeuDljTFUYbaM0i+CvnKFk297FcnwdYtd0x4GTzUk2uftD309JpTsIDbpjUCXJm01Yryty986lO+UrEV1KO6yeHWRrNPLc/nL78e3Pb39ilHRmKgYmaTsIWpiByodFllDMgiSoVAewSgXTYMbYis6hch7cvghqVvm9Keo0SWV+Zuh4icx2PWI81HDc4/o2gUIcAag/l/ddXTs4vGmZ6kJhewt0bq93iypNzg2SrW0M2BE/Qxehaf4cfOm0SjXWdeR0C3VXXFpMbhCYlWZ9VwiKydU1GfFzr061wIExDLq4tEDdQ6ePd1OaH7jiHgicnHTLm2rL3HW9uuiqM8dhWEfUt/GZlg84iDgFEO2jreXFRems3ZJv5KH7AFXNoAEaG0+ukMtSwpACoP/dAHS7uM8ZKVvm6s/IfGotYWJri2uwgtMsM4B8t9V7O2HTBQbdRh7y8n8YGdWhEnlIFIaEF8KwrfAP9eZ/R6WihaG65k1prf/uI1oluY7LiWaTnQ0GPSr/8Y6vE7buMEisWl+vkZMSnDeDsmsH4DHGH4J8Va/t7jJskBht0geI1rsmUquMgHkSS0EfdJugrCiDw8s5LbXaez4o41zf61l8LSAs76WFUw0+mupMX49xT0RdY+4aX+qxlr5E49pMC9F4+Stev8/HrJ9urVNMfMXkiyiLwE+C1ik5dVoBpk7TUjdNnRK38JL0Dtofg652Q/NcvJrCMCxCF2fUTntsl4e8I0oH96pXP0eZIhd+1Ofd/s8HeL1fb99/Np6uN3q42rZ6GZhrck8r3hlmj914gZhch3dyp28s3Pa+E0dwLR+SdAWO2wqq4U8S1m2zXwIDedBfzAk8zoZ5bXcLNi3I1/VZtL7hOToyEWGgb48Ob46wWouiA1DLBwTWcEPVE4rDY49V4HSSTxWngZRIFsfS4EHdDAvtVaNPoTN3GMJpj0MN5eKQxj+2hh7Fxhcyu7Z/yPgwM2ifxPMUAkZ6krfXdKcwzZNj6z1pXZGlORtdWZN1lcYq0FtggOsfKkzf1rdS1jzrkUd0dji4v7uDu3gabBhbParF2cX8ie5ny6ZW/F3AkUvaAVt7wFPvRlCWd/TykA0aUUcnaTDsIIs2iySyKrp0p6vtub57d4eXia+vEWe0nGbD/Zir60D3tTx7CPdU6qH70N5EDL0wg0aIFdiWJCcd6HN78+WmPbKjtiLXD9q7BNPN7VyBVieFsWdF634CoR+fjdCL97tSZ4B2eW18h1mCQidcjq/nCAZwaMDhAOy4B9kxf8yh7oLB5xhcnvlhQnUdLsu2X2NMzScMs8E3DEjSl/V9xLdDqwKzUgaaQSJxRGlK+sPlAE/PHG0d5nZYf10wngjv2hXO3sWxXkl3vjzs7G6VL4U3JsCm4TkQ18GNshKt6xMky2d3yIS1lghUf0ZdR5RR0sdOh8o4LLlpug4disfubuoZfBJDTHS0OvdP4YUnrmARzocL6SMHRfMuinTx3PaNEKmItTQFLDZlm0Vk+1pS7Ita0KqdGKEPQzwb1MbmfwUWywChCI3AT9NnY4RUCBRtNg1A2g6gjxPBIURQhMTMx1eWDrDW3gdKQwiBs0vfsw3I0f57Q6VCPqQAeNKCeGGuqqUnvGC+r0OgtV6+fmmBGRCZj4gYzChzkNvvyiL9ARyX4u8L+LlEUtGYPpKLixIYAhP/J/l/A2Cshj5aixLsvibQoHHlL3xALML6AJ8NgIim0V+cyQrEZPb0SZ2iBC4BZ3WaWTJJGT1VEtMSFNrKnHANxUnsFUNTtYZo598CHlXE+Z75Ls3YrLYNuuCBi4A/L/U9CjlxTuaqVVYsHPvE5jvHkpJHGmmySjsMy10cxWsgWZsFA7qSSvm1BlTUVHp5tIGDzBNFxTLH5n5bz6A7/xLYn27+68vtLzcf39ye6azw1mSC829s55sEOF2cPKbArgitG068kzCpuICvs2/eLSySsgzCtqqunBIeMtNgDgkMLeLitC12NY0s+RBSmlhz3RPH9CAjMViaRWfCGK/i8hk0dwO/3rx/8/bGXIqHn28//vr2/a09vBsAXt8OobpWUBtFSQl3fwn6E/skZK7gzMJIxWkaMPI8uFwwfldPNMIEYUgyD8MggP7I1qA8X5cTRv8LXiDXzaYqAAA=', 'code/mandia_stage6.py': 'H4sIAAAAAAAC/9VbbW/bSJL+rl/ByWFBKm7RkiI7DjU8nDdxBgEymSDJzh0gCARFtmSuKZJDUrY1hv/7PVXdzRdZzszs4Q64D7ZEdnV3dXXVUy/devHixeUqDeskz6y3V9ZtRf9PrCrflZG0ojyry7Cqk1tpXSfZRgprndzL2MrvMmvyyn198bO1zWOZuoPLNLWiaxndFHmS1ZWF8bb5LqtB/K5Ef9f6ug1Bc1cmdS2zUZyEab7ZSatI0rwWVpbXVhbyTLdhmsTMkjt48eLFINkWeVlb/6zyTBRhfZ0mK3EdVvxZhlmcb8UWr0WdbKWo5bZYJ6kUm0hEebEfrMt8a2G4MErDqpKVpYcLqziJajN4nZfRtaLlr26WGcL1LouImTBFH+u96bHFzEkYVHW4kTNqqWbHms646UwNbRrkb1P8mQl+5rdfp18F/t7m2TrZDL788ss3v5q59OkWYSmz+tRW3UdRvi3CqB5V02ok67CQ5/bg67erz1/Fxy/+xXgsXsnR2WAQy7Ulswjb45T5XSXq/GboDax8V/sPlfdQe4ultc5Lq7aSzFrY2OisgiZIW9ilLNK9vXzk9qppTzK0tbuDh1pWNejm8j5Kd7GM/cVywJ1K6kTTYkatTf7ilZWsrRJDhdWNvfT9zpyWTCtpzZYn4NI1XC/sAlIrMMPQTeLqZDHF6FYdlhtZ+31CjHMnyz4hJktl5qjZh/8+OR9b4IxeqSHw6s3UM6y7YVHILHYe7CS2PQyJj6WwVW/b64yEZXN/9VKP9Tick7Uk2U5iagh5gSGqIk3A/XLRLHrZTmNGVp/toOoTm5DfgY/H4cAivSWNTVOHBq6Wi3rZbg1etfuo2tGnlPWu5GdhVqhUAsMGGUaC9SV1JdJwBdGTYmBjd7Ly37tRmVdVAI0rYT+azl2neVg7Q5e3rMgr6UzEdKi7i2ST5aUMkiyW9/5oAh0sZbxjs/HtLM+k3bKk5nGr3daZDE8dNcIP3Guo37qw1W0RbJMMD4rtIkwwZJCCNYchRyQw9UrE8jbBnoD/UKzEHvOGKRnOKqyja6dHM7D+zfp7Xl9bYbmtLHkvo10trRW9gfzuwjKuLFiYtQqjG36aA8XSvVVf95HwTiab69qKk/ValpU7sJSIfGbLARtQBflHcixldQ3TdUYToVuq5Hd6HA7FvtM6PCJbFllQbXOwDlT23fFZV95bGWYkb+xSQgz7B1u+H84zuQl7TZp3t8zxMBHjIVYBQgxSQnj5rvIXLMvFaLJcnqivHr4PFFr7CjQB7FVeOqTupL3LhVbyIM6j3RaSsJc/+MXzjSx+bF4UQmeSKEyddqTGwn/wexTFU4rvDdMgyrPDNBQKyERBdvV7UmhlMgIZLrVe+Y16sYv0nffYvnTnuNMTswMjI+/hSxaX0vPhKT+o732VN7YCXFB+l9RawLE6zr5rKlAWetkZR71wwh98Q3JiHfYZDP5D7VeSQYMBojKg/XeGbGlkYtIY2abMd0XXypS7l5jRGc63si6TqPIfHhXqE8YpuySZqb4uPzvUV5k+ITf0xB/jBYMYAI+xC8iykQ7UG6jKnYZics79tG0HB4a94K4e/z+ZnC9bQ7esuwRmTTsMw5X3sDjvVmTUv4MgPCpTM2MnvjLO2+FcsXjiZ2jUi1zQ4pb+g00D2B73OFV0wG7+BHbrZ/J8yj2ohTw2O2r6007cL9SDAvN7koGezFUgib06pUH0W/Y81Q166xeP399JY19JCAgBdkXPIGdvT4+LuhXtU8EWhyiisGOe/TG6NEI5FJhAsIPJgfjob3tqYwqXoI103C5kuQV695qztnkTFuatk42KYdtSISiE09XYY8WSvDEJzmrFpKLRSiLewhMFDToyjfKylFGdSWzao/JLHBZpuVaFjCpBctwhphFxQsR5uRcVMDzESqUWo+i4k0C5E1HVsqh8FcfJW1nufYJ6GLMsy11RB2Htf4Ijpf1qBna3N/juqPiw8r+VOwlvn1R1kN/w03COoLf2mw6ndhujj6jJBdLNV/JZGmoiGsSNBeJruKhSYz0/u5dxuP1PtXwKU0MoJnyig8Cg9D9+ESvEp5XvuG+E++ZiKNRSg1hG4R5ei4CuAuqlzaA/yUyWIb4DECH9XQg3J2XszKZECgn5yr/vsP+3SYWA0yBroFGFoTCgaAEIM5dpWBDV2J1fQyxYHSJUXi/0oJTa4G1Yjz2cr8JKpkkmWc4DDh9JQixQDWCR5hPdYodakXkUCEqiULneqNjZepVVQMGD3gUdwUXweUYVKABuHghmWIY0MCUONZltVDvowQ1wR/NmB45RNY1EqYXqVrJWZEShXwZltiEatRCiwIuWil8bGpVdEVHTXuzr6zzTBCrIVjrt1vtCIqaPdnFIQEij0/f+FAEFsRiGWtpBaGeVi2t3VhzsrOjsrND7KvSmimZHRbOJmOVGAeuNSmBoEs5toFQarwEHzXztu2ZiHRW2LYqFpp15wes7LAp6CqOO6VEzhW/EVsB80YPmLOhkULx6bK8VzYuSHLd99e3y89V58OXq85cPX6/sFkTcDMbFKCHW6a66VpoFqzhzq30WORqg5+wNfcqFXSxtHUSchcOggBvkE8Jb6cQaSA47s4Y/dHTUa5HLjjgztT2VOGubj9YbAlUSrUejmjQmYDhDf/oQWoc91aejuOjbKq7XKnifpKu5ntHtTaPb5BYbrdWat+npNSgaheuq5uaJag6fUWiVny6QEHYMwNP2sTH2Meyrl8cqfUTHvI6aP6tu3qEBHFc/r2sXfVX0tJmcHNGGEetJq6yeMaWu0nqtXR1VX8+8fCRdmrlA7m0SBSxhBY8RFLJ5TxUcp+NnEGlvkOlULjXY4uHGe8Zmezr1f2HCHRksKQOHUrD38cdkNQ3GGFev/T/75453MXRYh0wlZ2dmUOOOOuUAtqFxr8DSynd4RMA0kXj4nm01Yx5u5yP5CADBeKgC96SNvxlhWNKMEGpYFeV0XRD+8mBTwg0Svtd5QFm+9nb56p+0WqSX7HcxaqAi/zGSZUyDtflOcoLsfzYm0SY/4pMNDDmTe/HSPXvpTE6opocoE8E6fSmSl04ymo2Hp9vwHjEkc0jPw6HOI4oNF0AaDjkcCVQS4hUbxNrwjQhKXjIDupMKbjPts0VSBU1hEoNxQOckKgdRe4hggoLf65xKIKa3xiROJWg97Sjc80jM3CR2GWNEpos+GW++yJr8/GnZw7BhIvL5ra+YfxnJk6dx5Uuehrk4wBNkN+bNvIMheM1PvY1TcRK/cWPEdNA+ljvvmlqjyhuyfv3lkNlnpIFkaEseX5f2jmRpwh2fdZaqOpzqDlrsFLTrbKhaJxk2B6kcMBrQYL2nBYD9zxTVXpUlIkwb5jpSdFajsbZKB11TA3I62tzkh60I5rRcSIw+5i0Q4U3DmSknu7s6SSuk+UnBdgODKbfBsdB54gpJHAbJmqxKsWhMq+MiZeHoCIyN6W9nY8KnfhxB+cSTIALUwj61lQ0JGAgAqFmlmA3JK2Cnbd3UUQRq7EUf7ewqZ/G5xooXCG5pdNYNeXuIlAbXhs/hILPY7orXsqd587pc9TBT3gLd+lL49fLjh3eX3z788umoLMj9uPFuW1SOvD1cIK1Q3i5s8LtJVqnUVSp69QTVf2wAVtlE6wqOks//VURn6TzF9O52IHl8shnNUzer9NgRUB/V/SmBTtJVVgTkq2TWTYOY6T+XBj2T4/CQnURHkX3Hw3QKB8pTs1TMOPy0fDaQ6Uem/9/jNV2GgPEl6z0nqU3FFtr17urXD2+v5s03vW8KjR3eJwizLvec3K43fnMC5dxiN1cBFaX9V1NYzdafXYhrGcaVfy5ubgP1dSrUGUwZpOEe8OVPAPW9F1OxXmcB9T+fibjMCzqAcpW2kfv0F+1RyGImLk4SMV22pyGLyVS/4fOQhydFY2xkCRctTN22edblYP38+HgY6MyGdEjELt5Pw+0qDq3EW2ifPiMUZl0jIgCYptiKFfFwYMne1BUtQHjUjyJidnTmQNL9Jum4Lyz37wwCIdsIK6uOGSzq2NeHm+5nfDp1zOiz5urTVqp4eL1LUw6A4QABIApl1J52yyWTN53cXT+zUfnNWaNDmZvqfqSCJWo4V4JHPRX85D0nhKJRJeHqIMy/0PWqWb9aNWOfjEGQQ2mGVZDHYTJP/YTAU8z0ufwfcsVThV3EwjAsydMjxbBjsLU67Kym/5Pdefr2BE8NJX/DZjlhA3rIe8Sq+zRs06CWqtGIG6EKxLq5U/wh+KMD1WW37q46rROZxkKd0VBHfRBn6DSPXf4ULTF2bAZwueAxW774UZ8XNyl5P4FfHpso1ANhqudH7GTe/ws5YMOX4cX3DTPEy/3BEddi8coVY1eMpu5SLPDllfq+XFLNlY40DzuMBZ3BLJddhVAkUIsopXzC1MvvzbGsODxFvF94wpssu8eHr4aiP9N4OWzPFyccpzfuMoxqQgk2M2wIHECglDlYJfUdBBjwhigMg3sy2my2n8sl2ilVhqoI4xjRWKBOKuHbiE4txZBw+cVMgUA9iNIQTi5uwBJRLsSw91pXZfyYcnBbTkC9QV9yXMVBuhXehglElkpHHf71W1fryXkAFaHbFkBD+IafPv/DorcQzG+7pLIHuuZz6B+p9DPsuE/V0KmdZrttUF+X5AvhUOZ8XUMV5ft1+IF1UAYhylP716svH95/eMuh6UjX/HQ1xHh08s/5jchTTjwhFKr2U+AvwzJNZElHNWfmJsSCQ1nCqcpJFYSkZDwOSNSMv+3yOokTmY30UDxdapPGICxTCZnLFxcoQAcwLPnmyIBvc/jPTqBGN2lv9edHn2fyrrmi4B9eWEE4w4cbfrMAnTDVtimqQypQdX0IMoQvWU3eTC6mq8lFOH09iaPX4fnq1Xm4is7Hq3D8Ol6vVmcXswv5aj2ejtevo/OJnK7Wr+An17Pz8WwqbZp0T2vs4r4a/6/V+/UwC3U+yPYDVqdn51z8V3zrSz6nLY1SANrhe1hSQlGO/2DHhLmePR1Pz4Ejo8nMJsMjnvjILzUje315iKdze99hS9A84eFYf7C5VBuO5WGnIKCwJwg4a6FLUsExKnwcEp4dJzzrE8LHM9CE5db21LEZVXi8j1/QKglZZlNhU+4MtpPfdXJo0801U5Ww9A0tvYHCWhNqWnymZWFsuihi0+IAdwA7dWuKzihvlebX4pbPuY0L1felBBeioNbNewSHWsEDdN/U13Zz7UjY2+QefLkz6mEpfMbX+3rEkM1XseaWO+Vmc2EKa8G7yZlZAeM5U+n7aAj9wkT1Dqn7WNNg0ZMZv05kxWtrCkZgQge57oQuAo7nFpXaxgKjnlh0gGrpM1nr08eP1ki9MgexI32giqbhXN+Ngf5ZJk4/VeH5qYrKMVJ1M6fDPwktBwdNccpcxTkxFRiS5aq5tUMCZp7zNAUjob672LuXFjThOcn11VlPclJdcDSvWbz6VSvO5hmoa63CNMwiTIZEil+g9Uff0pp14k7GotOv13ImKOQjiVM/LXVyxrwTc1qSTqitlcTSpMUX6hC7qMikW9Z4HyJ2FvYGiHN4hKG8UkChsTM2pyEq8VGUQQBvUpFoArqKcNQVXf3XZzijn68+fdMOqEUffXSkuZHOlus3HFlmvrkuIiDORbcQ01zHoGJMXhKCg0SXMoX2Xs/0yAkd25oReb9nKGPl/2K5zRUduY1OubGJfDA3UnRKE8FMT7L8xlzFEDbG6DXnRyo3ioqVJ5D3MFrbA4P8LWAjIZI2ITxs8/0JhykOydFKKk6JqDpzbK4f+RxhccjXkjSPR+nxb6ifLJPozwg5D3NX9/XLzPQmmpe98fjNMbYe9Xlip0SuQMfHYinzikM6Aq3ydQ0zpgPQTQ5iSl9TxEZRSHlO8rfZck53Wnp3QakK9bcpla84X7Q3uwRIabe7uXBY2dSVVCQk+r6quBDuTGXt4oCiHZ2ppoLNCWSNHhpSvuZDNLBeQ8TqZwjUKg+G6VE0/MyobK1plkpeXPxSvlnlZk+z99nUnM6zhXea+IiSmrvJPd+NsLad1L4t4rx82fh6fXxLuUGdN6fF2yfVuKZD+46vAijJb+kS5SUhMWneXZIhi9B3wncVcoDO5W2+PDm3spxBzaI0I63Rr6R7NNuCnIYFx5qkVqNbZArrMv9dZu6A8b65WcZpv754SckgYnMZaEdgIw/DFpjKUNelYReHXCXZ+j3Bz5sqsM/oR8NTMcicdvv6frmrghCnUyF+sFtgJDduvgubIxFVn1Cc2p76fBTIy+rgRu7VHZ2hCdexF9fyPk42EBDX8rWP9nVNRlc+DPB+70aRmunYMWHnwLWJncVDc5z/REFEd+e97UEB+i9Fk1ySpn1ckFjo8hwvLOFjeLXUXggSHMjtO2fH3H0EQ4NktcNqJzK3KrbzTeRGFCoQ812jkohF9kEUIpxx2hvWFNs93C9aLtmW20JD56IeTdbc0ntE2jH57ihPCxT/+ljPFTb+YERduacbptQsoI6mqklC81rx9afrnoU/l8d+vfp49ZaT2MtfLz99G327+mrCiAdboUSw2vfcqmJHH2NQdH3j3T4/sal6NYtrQ2tCFy4p7KqmnqAjFmpytgyzCn58jgMCjrK8fuSifspgIgfl4BXdkVDkkFh5W01+PM45Nr7yE96z8ctiDGvcqDtxLLSHR9EUwJubi+YnBL2Lnt1FtQ5Jj/s4z3c1ELji32scubrb6dyrIyopLro8GY9JZX1K1Wi/gradQEyVXVRhvaN19959c+mafmwx7CxdvDkfGiZdZAh0NsgQdC84XKB/w1bfQdkJ9hSXQncfPAF/FWubdl9ryfyoZrcrqUbhbZgZsNGd4Un+Cs50q/mkxsdcU++OYeOduuD9Jw/bjl0T7Dt1vnnWg+hGT/+McBRfXQkR9YGA/ioUK0j3n3ezavfoSI93kdwu4IMhzTaZ/bNgA44DrkZTvZJxM1jtYkqiuN6WGATh2oo+kleXtW/dDIks3RCjTWQgOhoqtVe3Eegm24QqBwv7wvzw7bTaZ/W1pHvF5gqyzpAroX5uMunGUBRlTVUA1dAhgn5Pd72v26QWWeqeXEYCTpB084/NrrJNmlTXc74Xcb1DGGlRRVXeYY3C/rxbUSt6/j3Md6nUP6uqrOo6v6OoPk1VOFcmutyAQO7JD/Qw0NvOL2OaJJ5L6laEKdGADHglm+vUVOXlABCUlBwJ+9s1wj5aYmUZ+G8vZFhhrePLEBoDDIk46uFFdd5QzSS/lWlNOccnyvdJZHdhZeWASfrFIok2XGM/O+yPKFeK2/hzrnKppqpCY4QplS734AgO+VbGzflqSlNdxnGifxzYWTz/zhBan6zIMiRmTjJV7jlWzeBY2FQ11HaFdDE9zfe03xaV3iGSFW35NixvOPc6ClZlyDVu/atAbYjKoAh8klQC8Qt1W4o44XJ1uUnzlWO/tPnuY0EFc6LUlXQ7CIo9G2cQ2MwbKT5dWKlVDkCvCr5Lgag2rujM07G53VYjqMYffPvr5T9+vfrp8su7q5Gqe1/pWvfyuajiuQ4IKoy5AvEC/UPTIC65jKXs94+ggKUB03yw6dhVVUsK+uWOOcNnJsiKV/uaKAuOhKl8XfOxOKY4qLjqAiCLVjs9nmXIgbC+kvLz5ad3Hy4DfTPl29WXnz98urJFL8NQEVPL6xPIIz1uQiYG7GcDpEeB2BAmE4RVlCS+Tlh7d1wat0mKMoASBFxLCgLfx/ZTtou999QJzOC/AQ4GwhIyPAAA', 'dialogues-prepare.jsonl': 'H4sIAAAAAAAC/+19SZPjyJXmfX6Fd114YXIqtlyqD2m1pFpZu6ZSKiuTZGlOwEl6EAsLDgSL1dYnmc0fmLrVRaY+dIXGbA5tc9OtOTNm2b9q3vcc4A4GQSICyJSrK7IjsLo73vc2f8s/v6f99z4Q75kwDvzYe5TGE+29Pnty8fq1P9X437UnU52m0aNETYLZo/ff69LVcZZ46jXdkIUqSvGA8htxQyrNGFfxM3DAj0OpIxzytQziYab4uZNA26cpk+LAJInDCR/5Qhkjh0p8IL6IIyFD3RXXHanFTSI1xiACJbz/+z+yRPQTbea3QkXCUyKMcbInvpDCoxFGStxoXO3TKSW+z2iAKhFyksx/MXQw+0HICL+IRAU0/DjqYRh0bKoSDOM//qf4smMyYaSYvvlJPBf/8beVBdGRrz1l6MLfn/2RDntxlKof0tXj7/9xebmKhoE2Ixx+7w/vfTuSqRjJyURFyn/+h/feW7myL+MsUHzhrhGsXOlLWmvFSzaM42Gg/is+7coYE3WjDU0MVwx8ORhcyDP5/tmzwdNnA3nxvrx6fC6fXvlPvDP/Qj07u3r/8vIStwc0g8hgDO99/PGjj757dNljWhgkKvJGr+NED+0HlcZok0pa5DShhbPLKOKBeGGn+48iimmiWSgjcSMDTSNWPp5kJ7nypEnWxw3KF6lMhirlT6j8fxRJTF97kqiB/kEZ+lRhfIPDYRak+pGh70tDUvlNRlzH9N190Z8JM5H4DPSuiEZ1YxdDTZX/uj97TeMhWqT3DmRg1L/8l3+uCo2zhqDxsaWyBrDxh2gJy99kHRrZI4/vFAZ/iAnRAr3u+S4M6ZkU5s1PPfFCjGUUaDF+87NQYppFIIy+CjT9PQ3on3EW0SA1vX0a6Eik9HQZDekv2aMH9bP83pnCqZmiJ9ClNJNeKTjPu+KiKy7LINoVZ6UofSn8OOqkYhzF0574ZqTEdUYzNVL7gkhVTKXB6YGiCURDoVMayyyME8WM5GUnFCam5Y4GmVG+PSbSUZwNR6mY0u0qUcwEZr07OEC19cOLdi8hzty5io7BtIHBnDfPYPahfJUbfKoE8RNChcFpItKAFpQIVoeTQDFP8gka39PTcCICTzFYV0lHJ0FmRCBFOP8r8SFvFBvVwwNNRk/zaFDB/JfUPmSiEj/jkyQ1+W3fZ/S4mB4wvwVZGDFSWUK8a1uOz+SYGOWMBp8xIzEEjwEhIOoMJQ0VAJjpnvjQQmY8efMzgamcp1yVcJOzrlhhN7t5CnOFJJl1BdFI7yUzFzGin1mcMSvpiV8rmaT9RMkxrYM3NnvZw/FTc0BvA9Avmgd6+wC8wV74Yf781sTB/JaAYzUffhReQbTkG8F8ir5flg+PJvJbjDWbpPTQrvCYgflZss0evpHjCYMnl6iRJvTQs0JSuqPOOKP5zeJydvC4hB3kvKArrsrZAY0oVjTUVV1DTGEepDEd2gv9w4ftoN4GqF+2Auo1g2kVql9l9jkr3IQeE2taMZ1rxWu4+4K+sByRhOqz7kryUHw2oWOhhlbLqisdl6BkOvPmJ9Juu3xoKmG/dEU/JgEXkErMAq9PxP4pjTob2MeN+SoYOLG4VlHnR7CbrBzHT7riaVc8I5Pg/RJEX3XF41IsfwrjYEhUA1GeGBUMSNKHmEZo5/5xMoOREGep0AOW95EC7cZ89sOIfh0pmuGU/sXpTgJWK33SFQgvCbMEULLIXRWvRrAgJP0Qn47SGTAU0yMSMaBRCSj/ZKUYJfdyke2PgGeXfgc+ecinwIUHfg3HntrAnq6aZ093MJBVXvO1ZVQrSgqR8ERlcFqQwjGhm2kqBBl64tcB9Aq6FgyNsEVP7StDy0qniBt2RRAnJud3RiV0epLM/5p2wRBD/hbzPwtJyk+CpwcdeSPhTHkZiBnpSX0lMy/OJsLvyCwlfglNx8RRRL/RfaAvmmGikp3+RW3Ysv+GbPSMoUMzIsoy/DvL8IGmtYkIg0SOQrGHpNQTWeaKfNwVG9ytxONhfR0fEYta00iIX8FyYW7EmgkRvZBD+tBiptI7/RdHztHxhTbwhcfN84W3EeybNgzxM/hfiatdrzM4Gun1Ui27pqfwgNYfB3UsIBOM5p2/eYObvICYZnvgxYCwAiqEjw9mHA2DBO+bn+ALGNMFv7LbCSzvIwlVQ1xniv7js6W85byEt6wwFvopd6e+IqumY0Q8lrOe+I60H8tfRpI0GmIp/VzZEYSTUSrkFJexTpXKsdWpWJUSRM9kKMqgnOvUuRaOB7WBBz1ph+lUO4ZXmcTH9MjCySqGCcGiJ16RjdCR9h2Wi9Gt9GDo/PyCFNaEz69O6fptRpkSgayySKAsVGm6yx77p0h2hbW2voghjkNNgNjrCj27KPOFnhEzON/HDKKxsU5QUjXkhEjO09Aq9H51Yv8YHVrbgNanzaP1wcG0iuQvVOLpFdfKxwoeUki5zBupXWaAD/EDQoAcKsVa2S7m2Tlh7aIUa1/GNL+4T3O4Q1HfHIVDU7NoKiaqTgzFWTznuHCchoNwvuO9rS4Yvd3T7kt4neyutvDZ2T2kvzO+xMTsCseRmRJTEhEZrh5CuZMDa34OI5rSTJKWmOp9DsEyN2Ap0j7JfNUV+c7/yG7jxWNaGavTsm39xUwMdRIMEloXH/R7TgOVibHBBXy9IPtiqtMRLdlexD7Iyjgm0BYmcNYgE6ga8nNC2JyLmnunibhJUXaix/aUzaWzisKkhs0dt/vytmGjSQ7/AD7Kh9zxcHscDn0V0XfeIPoadv41t3vgtgscnO8HzhdN+kwqegDr9Lw7n/vfLclfNkjynxBDRg5J7d5t59d+dyn2vFmDp9aw99Mz30oy2KpbOEfnrdWYheZA1haQXTTsVShLBVkHTHnK4jYkKqo+W6mYu8m8ZASOkNtCyJfNEnLFvMPj8pUrakGVM4pd/u9bQ+5XLeDb95h190BJuS4V12HucMw9bhBzVdPn7ilt1SWrOjBYMDxpEAwVUj4OyCstSfisioD9KZ5VUiwdkbeFyJ82SeSluUrrNL03G3ebrp9Wo+s9ecm7Sbp8OI6s20LWz5ok6+Pz7Dbo/tBE820MPKuGgYrZ9yW4OGi4DiMPhpHzs7PXryeTSTCZpBMa3iQ9qLTl9m3H5Kfky4WvV1Le8nPFjtH5ra+xMojPTjVBhtSeGF8EGw+s+ugkmd/S4wxhbcvLZMjoNHIyls/FZ0RkxJPHAeIjYG+u/QGLUzCBjplQcfbNv0tWx40UKZ31Va+eGpdfc0ClOP9AWLu6a+M0notXIwI7YKTpWQmxDkNMYCpnBl8yQFwjnyWbYqAT7Frgti5HcRA+ZySabohjcBSIjGbFBqApg+Q3Cce8v/n3OPpA1L9WDsvNY/msQSyvZJwdB+YdWeJdGwmGWHt4AJ7zk/MAZbrQkCAK+za6K7uGGSJkGLMDWsgMHyfUhJBEDWXiC/p2OjD0RIzhWobwK/jKwmb+S7jFTqwII0i++bkTcsRKHrBIvwZ6rGyFTsJOl/AZhBp/iyjr5tklYWZ1QroryqyTmd1hQtPLo84NQiPHk2UsZCqDrEyLRfms6mU7c8Zz9oHIRTpv+xCGwGgQUW2Zjd0OShOiRVua83OaXBfsJ46QxcSy296dkQG6zMvpK09mRvG5YTbrsMZOzMqm80zpgxUsjDN4+DKdBHv5E63hB+KglbfVd+pdfMfDmudh5+3gYc1woFUe+AKJhSbHbFrwzuI1SyZKCrkeaE+rwPTE1zIR6gcFT1FX+EgoJFwAIV3OUrQIJsya+e1EJhy/IZN85yJAUoQOYIhje0P4maBhz/+NqG5+y7Eei7nBTAF66OLN8FmC3EBHnQGwT1jtCRxZptvFbC0DewTM/puf6L9ete2/vLTondpWHjVL7C3RdHMWxBkctjgG4DAjSZd5VMytyMoJ6MxAHaREHTVXx2Oa5zEX7eAxbwvC10pz8C7mcmw5tlRvvc4AezbgzfMQ7muYexp+gY7oKI+bTiCFmvOid3ARk5F2lUGuj619YkhyQ8QrG6PO4r0nPlLC5/wSZNlg2DL6MbdnyHCx8VgwY6LOj3GZflUUMyyvXHqnivXiB+mlwazHuhYZZpH1nrCh5scCiEhj+nekEtKtwISW6hfJk4Hy0qKS4ZZlN4DhZ0Sc0EzpMySKl97crUgdvox4dcWVdIyseUZ22Q5Gdl9MYU0dAsCULalAjNOGm1rHagdaFQlhzUGpAhQT9sEeOSuIh1YwVeKFA3ojipgZ3l0GzZtJlqqlSkbfHGdWlDQCMqI/pOfFib+DVQFZ1gwxGTa1xJhIZab4+DhAegBMmMmbn+i/wotCMJzCgESaHKJHEX+UR5V+trjlmgl2rMZYyxci/c8/4Yn2AWqA9F2RkYYZcQZOX3JmQn5VXuMg6x1b4RXVmsuLvK54tcDMcm0qtxetfZkmMwTVpjFThsiMVbesy2vB/5Q0M/ZprXJBZCSmVn8Dbtn6lHlg8ef5/SOo1NM4Gdt+EjbbySwzndRgEFtfdT/TgS8k0ZZJMaKiwIYZ6cnBit4J3xjj2/7MbMDu/NI4c+zHdly5ea581RL1sjUcc1uBXIgCKMCAFd1CV8xkUSKHo0p9neSKa4cV0/WRDaSn6eOrdDlGTtj2VmVOwJyjJz7Pn+ArfFy1WvjAMgrDhS0HEnmkWR5XoTAUk3OSPMyCOKVJkwwjy6JlsR4E9sn+ljnMoW/M1F/CF9WJNLxTzD0Y+Zaj0FIgC1rguo8Uc5bC35Vfx/yVrUr2a+EJ7N1is7MsH7woO1dSvvuQQrhLBbdIFs/N6ulotgh+msrUG8HP+4g+q1lr/cPqLIeJTFQMyySLYGMTV9AemTETYviWFQvZB+lBONyt11Ze1Vy9rbywjps2yU0PLfq1n6NW2cAv46p1bE9+k+glbeaPGGuR05qlwcOQXbF+yhLD/82+1RMfx6HyCz2NLNAVVN6NvqMm4pDUDiSdNYykNuzsNRwo4OIEHCNoASNoWqbeg7Zdq4ekapmyuj0QzgPwlgOsJaK2l5vcFk/wvMKivs7z+CFAlU3i99VuHfdUk7aecmUnWaL1G5QOZS1B2XnDKEOtDAPyi3irlhTUoeI6zkR+KZxk9MuqVSgMwTD3Z20gqz7Xz3nlCpu1eWmcm+XtBVPjWDoiIOKIoM1aisscFV1ZT+Sjg0s74HLRMFwaD/Z7kODnihVuGopedqHI7ybGLxvG+MPF291v1K4L13U4PAWHV02rprUGUxwb7VpLvZ+ag1ZrDEJ1YGsH2B43vjVQU3BmHcHhj4/wpZwWwO0Csd9K1Dxphap4P5F1tcY811KC6B4DkB8gIthhth2YfdowZksDVjdix3ZGj59c7qg0zL4EAdujcHTcDjp+1nSA5N3h1Ruuw9LMh5MLGO1IECnx2e0eg6PpB6Ppy4vXrweaNIyBpp+DKhKt31JPt+z1wiMo9kt2cUKfCCrQNe/szP8t1nZXN1S0kKj9ubg2o2/EDjf6NNH8Fs63xKpfcK+bLrZ82RfgKd4sRt9QPrXwGIgRrV+yFWjxI5zDszggfstdmGZxrnCoSGMsRud7rG9+xg+Imft81lO46CXCDfmLJh8I7jjqJ8qWBCaVilYIyhUUp0GWRBpFjjEmPYyel+EtpmWOxDQztsku4i6OnqJDabMoPXtwlG5U52gOpqvpTOvdgG9gW0kEfugEb6CrWaMLUR8f5f06KyMO0BicPtrERjSqYsibbYS/7CDIdlElWNmGUwwQdgnYkno98SvGTX6ScLSoJcy/RRmeAjOpEy5CdNlqCiGuC6ytAi1/SKBWwjfsnvSY3jni5nDWqsm3qAeSDEsrTGutZPQriZARwaseZ74mm69zs9hihz1nrbc4CQue5CVKMoNKR/Sv1T4+zFlWEMRTY3uQIboF5dpFOOMatva2GxXMujtZ2/rN2D8MZ8VNOuVmmSGa7qEWLr+ZNaFUD7Igf17k4XPL0o3EzbkylzyIADDFltGAY9LNMunzJpn0vbDGtSpxHalXEkQ0PYrWNKOH4ZPgLQmLAm5KhRxQeIjmt/O/wK2Kt5DI0CQJQiZpYiBoCJp7ycIOOIMObU1W+gR6/gvdhafyF+ruHebicRxUa0diWwLnLbJEbiNx0HuHGMSOzFH0amOThK6MCv1omkWBnum8LedYjnWuL81i1qlwJaILomyVYewA9vM6qy5tq4o2sQxNtgfWrzBCgdsIqT2oKHsDRuER8UtE6UKOxwkzSLnFclVytz65olDe96o5ltYsS7toVO98tzjOuiJrtV5fDwYIv0jntzTDG/XjI+bcRW9zenARbbLCr+0g+a99euwSm1aPYGDmsIw66QYooVXMLOxY95hK0i34H+JLmaHXYdZR1hPf5LqHUXQ3UdnigZx7xk3G0c44b9NHGg23KM+9PiFHhXC3PtvPlRtACdSW4hYdME1txFYRv8wqVe4y4gctxoMtbftuf+FUQnJOJomzFIrWCj/ZU4+qK54cVZJqW1f+KtrFhV/u5sHa8IWB9Ma4yYfBY6DZao4LKKY6s0VX8Ix4QBRMy4I1I/TaPRZ0FCpcyrYtdTqb0PvA5S1LKeLv6CWkQic0EOyPktIcaY/eZxV2s1MedFcHvzYtPI17FHE3xwhuCqKyQD0i0OsIzG9lAhX073siXC4Mcz+0i0c/APk6adisNLxsVsGvX2SsiqTPO8j59DffwOlrEMT7XmBY2NLK4mpfLRgZCWwOhvSAOC9VWdIjBknCV5K6mt0se93aFDn0Ysj7T4FnETnRShDYbTYd4TyZ/++A+zdYbsaSO+dm3KAHveIsXSnL3OjJPdRfCNXuJejauaG9TwbehWfTqBKoBXmIdz5wwUyNJlT4sUZ46uZGy7fgO2jDYDVhBj9ijYkvWE6S87VMlHuOn+4vEZaLqrvsgl+jp5y0HYME4SbxuGHQSJlVwXS487jizByrapZVXTXJqhwrOYqVrBXu4mY16QqDN1lCY8I6eJsLu6X67wZmoWwEC3VjIguHIv5EH8m9SfrP7qxvVc2lXMqWgAzsJUOx5I1lOCtsmN9Q3+BoNqmkUp6+II6hNcXQDi1MVc7U6mlJ24Zd6m3zJ3em7TAZRGEe9fI9jQEdtNYQ52kVFkof2x1vfh5yylQ9RbB24T282/XYs5cFCHi3S0ffLM+CHgQxrQEtnDIT5dGXC2ZoQW3yPBWaRBwWZvOUa/dUYBAPsKyOfzTPP84a4R8Psmn9MMErLnbFIe8o5J03I7mrbSAcFQhSS3mDOkIuag9wcLhpHjcXjeDmrdq6fxsCsSpWVXg7Qqdc8JPjVuvc6rIh/foh99JbEB3kwoIc2JsH+1UzqsnDbC08UCxNLRUmHiwGpQUhHA74zQP/cSPAf/C9vs1+EvcWOHRyEYx7DQzbk55/D8vh8N08vp80gu8HDNJpKtC0Ys2NJuIxXRDj3wXEnzYH8SNjWu4zsM6F0znQHACaZ01Fn0CQSb0Ei+IybPRYUnuDAN/Bz3hLC71K0GYiQmyKXBRt6zUcHuaCwhwsT4bl2fnr12G2/L+Dypds3nRUpGuCsyUVTNZjO2SCyvNQT8n0uYmt3vZcoGmFQGE3aKr9JKNf5rf4RvQIlnuKdDzFNffDySZaf41S18uOKV92iIwHTJAiiEnFM5r0QsVSJaBjcqajTvlGVNWqJL+NvEB9IL5TctRdtIKBVZnmWiY3XOJKcIqQSkYjesnFSULXlUDxFb0dRd2OnpgDXdOgO2sCdKW57vWgbjX4+qtMd+FZsg2dIIE7Kgznv0QQR+rHPFzLA7nDcYEyXDrawO1LMczgxITzw5dV22DsKc/xBeHxpQHyhnFecJEsNPSNiYwu7xTzBeTr5pgclJqG0nnDUDqO1FfB8qJAorLl8G8Q6zH/G9lOeEKyaci9EEWYALP9z8HquSw1tvz62ja0FNyrSLJnkeurQhRwRB/dKXsir40YdQI5IPXNKm3KEvaExooCoRVDMVByYb8QfJmKkH2ofdXjNqGe5N0T+so2BmMik7wdDaunKZrRINph1hOo/QixOaLPymKzQ+SVmRQeGh3xLozJ1F0i8wHXznGGpjnDRcOc4U5crzKBr1WWPpr/NU0IGp8vpTC2XGk50MzGxpEZMdHzX7yilbf1+Ggch/hOsLmT9ATXab0mGT/hWDI7hLxq6/dZh+T5RKVosFP0HN6umExGW5c7TCCqCBYcaGMK1HxqKwtz00H2b25qm8BCOf+43FMnYI/E/lYFQZckduyLIPPGKFaRsmuUNGoxxV7NmhJt5Mzsk+Q1zdChvGmUXzaM8qaBu8pEPopJgvXR2xFeH/oSPfGJNo/CGI12+GHY9V3XSLZ8WyEEGczJWYzwfwjMimEWe8p9FJqAmNLKcE/UaGzuktolY3LYaw57h6YU7sNfJbfuNgbv33OElBlUyCfxUPhUIvqqz4l8I8T2oEU8fU6O0UGdfSsrSr27FVMAWejZaufZxHZ1m3FPZJk8t00EOFYhtH4kskqVJp05irVBLFKqgtk++Xf83Bzs2gC7s6ZgV4dr52Fcss4j6zBVCVPnTWGqogfoNM+o84r+nZDzRVPk3DJfRkv9pc5X6nylreIXl03xixrcFk04LJ2z0sHsCJhdNQUzDoQTA0VnSKqqxNP34Ptzfr93iYwvzp++fm2u9fWUfvT0um8kftHXBwWPld5dexRZUTUMDeM5Qpopk/5j6ZFmQqLJ5iQDD05UrloKyXVN/M4ILo1PY6Mmo56tmb54Tl8ZWk+kUhBF0lP87b4HhZbEsfr8Bxv1YM70VGmIaaue9bcZuSe4uvCaV9tat8P+QHwYzWwVAqQmzeJs+Wq8LfcjWssuLq0aZB9mK5wcMitbtXllYg6arYHmWaPQ3NH86iGwudZuJZd1hihXh5M44TeSTslJjWmsV9C5vpdgFFfPU9tVDAIb2rzIzWWzpwhz9nVgHXtAO+wdTuUr4qEZQ75eXq7Gpb2FUQ50f2OT8h22fE62YOBI3tjNgUTZen9xNhyRFYek7jRvNsmF8W2Wo4GZrgd866oXSPYDroIyVmqCLMfruF8mu/PXMw85cLUsFzl0wRyLaQ2LOW8RizkZ6husA8khKqETqJYwmd8mOiZGs+A9C18VMaoikYR3Q6Kc2yGX2sz/xmmHGMZNrBMxlIlPvA3Z0iqcBPGmAv4VYfM///TmZ/5nZcz/77+TKA65jfEi/98CCwbfIcoFSgt3xbM9G/a2NvodWgYNMIrX5T94hSF7M04SLoo0UjLh3OmjFI+SFcCj7lgEp4u0l1FctIhRPBCw16LhgZpoBc9bKpE/vzVxgDIq2E+VE/qMkZ9UUEx8bgwNL5BNt4ISAt8r/D8o/6XpP041XkhTyNkJ/En0afOaAnRjT7zA31r4ubmN65StSUb/2FsRHBDmVsFnNq+Lq8Lkj8lC3viNuT5BuVvtDCH6pM6ckT5zdnFnYfSce+3VeMBq4P8iXmPSOO/6A294XxrtQQPqQn3px8bAl54rRVk0kDrhgm6W03wzUniQrR1JFDq2zAtzhuc9j7hAaaUYVeJsxeQ4MSkUo7x3J11n+DHAtbqhBclT7PAQwS/MfZ0HKFI7vi6P85gPjBurfWPccdBndhy3NRz3si0c9x5431qXB94GBCsH6smeHNFHJ46s7L5hFxm9Cb0gC+e3XRStmUiyMLHTwLuRxRh0hP0xOhhxNemQGLgNHTMdhWI4NKJlGdz5n3GBsB59nx+GQvTK5FuaCf2bzv/ijVDP/gWhHw+K1PLqcP7XkGfDA8cgFhsfm1uXpOOQPsPm05oziWG7YTIB6OwlzVUiurOM9ZK2d1bmwc2ZLV2xxaD3aIavrKKXqiTR/dXBso4oPEn6GpmSgSZemHPGtCjAi4oA7GeyauMdKmLJolj97+B1cbyqNbzqqi286l1jJtuMchuerNp28uhZPAVFvhJMGsEV/LaUlWaNRhI6MtubRobz/9HW0lai60C1yDcjA9ZcehwUN8tL0oWExchu7cBSK+NRpPedES86Kys6ss6e6GfB1PbqiDbgb9VyTVEWmKgdTTJsqwyd5myLlpjY1squbUyHDtHY8jWxHRNLl8XyrD0r47hUa7jU43ZxqVqQvOHzsiG9HyMCEbE8FhdifhvFCXE8THhhCRflAtnLBmtmU3VBVEKXLJN+TP/ZSnVknJDBlJsNKAtCBsVYLmpbg6iE/39uS+3FMu+VBT79rHGMPbqKjc4YaFhsRL4Sf0EHiZTiQKoUdb0l6aNqwuXA0pFtGCCDEPZOdKeKctLkHeZbgPlD81sOwv2JcRvfeBxd/4H4RGU/CDLczQL00Bss1XVtDCV/r0VRP08O5rebdTglCJTU8ZvM6sOr5hnKSRbP69vK8ZoIVC/q2NHFeKpabNycnObyDVEiyeVX01gMEkKHvzI7oK4YDpfi1qmt649QRi8eDBRsibi0aFGdc3WwbBUszxqHZX372ptGP29zDKA1Lkvb2aqSXOrV552R5a0nZ8SsbRmTRjwMpM8C0SNi8Wn81jpJY1/Olm++SwmuNg8Hr3bBq3mxV26NH7D5UbblcCwwtncWjvPaOzJvF5k3L0ZO9CxtKndH7gjtwEpFIVJxm61MYTtm/A5U7QLVeeOgqsXduqmWVd0z3YGpijmZp25E7wZZxZk4dLULXRfNo+uUPc0TAkaqNgE8OQbDRT28hfi4bInlsnsr/M7N/bId+4qWy53b8kfusjtqbxe1X7XBD3biBvKJwSwVUxHrCEZxgSZvLWAet8JxvKupxErMwK5wiGNdV3nAw+ExC45i20WxT5pXaPaFBGy2+dsb7bKDsJ9U7dW3L2qnrNle+aActbeL2p+2QqE5JMTliJi2qg20qgWguVCyt4jOnzVO51x2B95P4XeIgyZ+13pLrQtnSA/30bzYTHRivw9WVwahomnq5sK9XHCXQ2FdKGx+p+KIciORvwipmMIKDXU9FSBfFjt1DBWuNTxSwWSfDbxrKI7EW0XiF23Qpw7Nzj+qtk4tZRcPqKNTU80ch49W4aP57YJDc7zvqvFQcYvgwNIMrqZCe2m3Fc7/YwoqHFlXqZZShJULJNVX6MjBp1XwedwWV9PRBYNOrDP2uLrrtaYyYa7e11uKmeY3IzYrShxUQqukKtZxNsJmFawja1o5ym4VZT9tX0ZRWWmUdZI/oDDaNuk/PW7XbUeVt93Uf9egHPE/GPFfXj5+/Xo4Gk6Hwyl+hvjfQQWcd955H8WbsbdMKo+mz4Xsua4I4sQgje66A4PBsDt0/pchnYniDI4k0ttjulhmXsb+UnQRIYDIROkA/UQmyBbS32eEoJ74Evf4yt5CNJySasXOp+s4U0lJIEZXfCmjCL5+Lp8faMGHC0V/Aq/neKJFuXp1RKfTf8Kuiy/JCElVUgRvdAUfDmVPfBdnnRsoWZyXxwi0CUnc2yMIQLewXkqDvz+MbIC3DDKe1krkR6UJO/y2Ar9njeF3vapiswDeUceiazcwH4VoGtQTrzIRdsjCSiTwSBLVU2m6TClEsnwBHxrLIN4Kz5ppAETZqFhk8/2DeDEgwmfpNlOinwnkLkW+FAQhFns20CQDapFD/jNTnyaZHtvjQkVavPkp8TWRxpufk+LGwcq9Ipj/654i0V3xZA+PKS8SbflJnI7AZOyOKXcJSpNM/QPXbUVbId4vzYwSWGky1oYYKH8CaACS+NAg533fokWLj1AFPpznohh4QnwVkxkb4utim5XIJ0u54WYsgJokJcqil5QaisySRD9AKv/2R+Bqhod+B1x86qdwfK8VfO+8JXyvHlazyr9ejESf2GhXeJ0i95N4zRoD5GAkfJSMsOZ3VpBpmSJzWBkScpOcdS7YLbxMKY6A3V6jnkgOA3xyOhRbCCjkyi2f5dMHBM8tyTaNOuMgXjxoE0nATifsiY8YtzmU9FhVNEzyAtU5x7tbdeImaHYrrpQJcWe1WSfJGdlhGtMH4tR5Ox7SCh5y0QYe0mq4r/KlvA0jPBOcK7vxtGD9aT3xMsjbOmJkRF6bCtUX8P0qUBUkNxfgCZeilnDSJeZqcabsKQTwDvuyuH4pmOkHrQUNaVqf2UojG6dg0eT78p2wiMriWiTCIDYrG5BOQFjdzZG4FOqd9akP0bQWDnR0/h1CRWLmg0PEgLqkvKRrB5HPayT2oujzBPamOEtNERf07UhFHNyGk8UZG+4G/Kkf0OTL2GRhHORKLpGaiomKJ4G6W+fCZ7+Hz2QrSp/0pRwPbQUPvWwDDz2dNW0YkBHMmSL6uwN7FLt0uAOHirtQqEObZTNanLP2q+1Zqax1Sr+MYrz2NxlwmGZ82OSHuzgwwHPYx2sh63FhNzoBQqPXJnw5mPMNEuroNaS8IO3a3OG76q3oKsjpX9FXCv2kYvWPojT0kf4rq5WRYU8W4qZudoLL6pB5Oo7RCo5x1RLL7S3C+N3urVUOuKh1ssEKt4w3bikbdfoZB/CmeUk2xHqhz2yUCQjxUud2aS2TReX4w31PaCBPM4SOAtUn71VNX87PPI7i52DkoYqUzUQ4yFlUYYKONbSCNTxur1PnIIDtdONcY1N3MonNoi4k+t3rkPRvu8eLhRbD+W00vy2oe0Om9zOh0YQdcWVVa6PYCumHSmsLQxNXEsXrw3NYahhLh5YuvhNPJ0Yz/LZIIrMY4kp2/D2UrU9Mwo2ml5FOTsJ0/mcUj/NozVkqJtqLU3aVTrJgK1x0yeVNZpOr+lraigtjRXhDPpaCqTnF6fm/Bpbp57UY6ylbvCLFUOOuqExMcgVHktjW6vJGUiddUoKLwsWCJkErmhwgwY6bm4Nfa+B31ij8viEFSqWaxjHQKMIifVop43GB4hUoGlKs6KwnIzmZ3wJ8ieKoU0EkmM5vg/mtJZINEL4kZYq/G5FqdykQuvACMcn6HNuZE2ukITVUoOopWPxrgthwVXbBbZQqFZHspd9X8IhwiJFCIWNv1BVTmXqc3PPqd2UAPHpeDnjtAV6zgu8B/Dz34eauWhz5npzPzi/8zsCwWQFYm6fk/p2qzpnqYHY8zM4bhVl9jo5780g6R6SDVXVYXTQuvVbQFGe6Vregcwj+XdL0ZaM0/blccwhiHDqFBWTQISmkX+Z/Lnx/m7GGvxerrrFEB7qwAuhBb34OkK1A0gZhKj+t+MZqKqf8+1V3nwl1kEcSDmP6pS+9Mae95l6+MqwcPQWHoPYg6KppBK059VaBk/vtkjscd1sce4v0lk4t8cd6Si6viZV10Ky65P54qJDZP2QHl9bA5bxpHep+85LWkRV1uAYaWhQtihpge0YOMuR9zxaZJ0Va3dCSM5cHKQ+IP8WqecnsIewT6qZw170kvBlrxEjMebqRtxPNQJo2dWcgI28mhtIH2Rxk6Jy+AA66rYFus+ZPpYzd74hY0V2ACAzCYD3h6+TKcMtUuGFcuLlZ+YPfuwwX5UNyJN4aEr9soX/65ETV+09fd6nrDoE1IfCqHa7r1dyru9K+a6lBtz//+rhMaEfWrSHrx+3RnaokD270J6uSVn9ybbkqBQdK2loePF6HlNYg5UnzSLmf3Ph7KfnxpDKm7qkqh6ui8c4g8GmjCKw7q/xBS1K4QhQOhnXB8FnTvoh7yYd8gJIPJTUbTnGa11mloakKCw7aDwbtq/efvH4tUzONUz3V9L88nXGYxGSAKP+OPKyt22vKafxNpjjg/FFKttAkmf9iRNCZ33pYqOfb4eQzW7beZOMg64ovO2hkK6ckVlKdLgIVoo4t0M0nuEp3PdlUHw6wMWy8URxzFF5mMi7HPySzLw6tu92PUVMcf+4ry3/SVBxo2gOas+ZBc6M8kUJVC0nkbUDmd1oFmmjtyw7v8wREYqGObPV62YluwK7DjEs/cmH7RZTCQAd0TYiMhuhHuoik51hzWp/9m4h3Jm1SgyViSSfLtobzQrKHA+0bgg872RHbx9tZYqrTEZA1SDSa/xX1HENN+iuuQc88knBEG1qR/I/RsglBsTRu/ouDZEfSCj10zTD0jjhCU+0RGcilMvDAFcyLNtayiA7f7cH3eeP4prWZKjV+RFS/ie5X//knHSEqO+qYWGhMvmj0aDtX0M0INvrMypK89RfIk7ARaWhjqhMW1IjU3IJol8cglUirnt/uaW2DoofVAP5VxJMCkiFIYxKspF4G6MhBM4KFwK96ZaNBFiA3o3gyIcWVEW0UOmQy4PlvywQkRquysAzPB66Zrf1V27I5SLcH0heNQPpDiDPgmdumkemZkdAOJyaX33A/hhwsudVQjUREykQLGREttcF+LBCxVJrHVbVtjqBVzbsCEp2bCRgOt5haiF6McFbeK+eAgToctAcHl02KNlJYH91gYwKfKkRHJhWVQaDQq1ZDlcgqktDLVlSp3MEQyxWNawYPEJwQ+AVs+lV5dn/1bYuFXIJKiZ7nOoqQ2U/CiLON2YW61D45iFi8+h1dSp97VF5eo/4JO9y1B3dXDeGOty9YBOW77dzXVqF9T8oNe7b9Lav6z4KaFjoOUXyxEaLiwgg6uWrGms63gEwBqQNyh6sP26GjQXQcWghqLzBO3Fj4cN3xyFnC9F1SwkSRCZynB3sj9SPZHBrbftfWSBMh18NAJ8RdBTCcx9IBrCUAO2sMYC8ik5EykwPLVm6ayASBwZuJYKRM4Sps0AlJbLq84oWTTw4+Dwqf5gTUp0VbRaR80QcQfSUzL84mnPs1ig2KWGRowasfhfO/hlyxKS+aFsakdnLT9rBDA6bhq+1g/Iy+Fpv+YzmGb6zPfm5bQizC62xxP5PBYx3qmoozYQ/beh2UTCJbejAg6pF9WFThzKhg0GUhZY/Aw09CKqC/WZsm0Vvujqg+J4ezduDsrEmcaTGa/yXlIBEf6mB2k+gEoSSIHAk68oZoJ0FqSDK/NSq52QYTopxsWiEKfzEFTmOBiItOWJBkiBqZ7Ez2y9P/z6qjSf3gkZzlCGX4x6dIXwFqBlmKdtgj+gblWZLHjNxhphWYOW8MMl9bNc5WVoqT3KEd5hvSvaP2o2tJMT5kE/m0PV9H+62g/YvGaJ8j88mgtnH51pyRmfB0NL8NZRe/02GvC5tmpcYYOxFInATfZ50sz3DZ3BYq/OK93b7wewnUuDjeA35f4RcupuKdguplY1D9fBlG0V147lB0A4AMuOQTAVCENGsC7ybsKkQNnJyhfEhsxGmxDQ4MrQDDVSvccVxfphBedmfIh/LmkelABAi3NtL5wXchxkAN8Ckjwx/xLWqvU67+WKOrE7x19UQQubCgdw2Ijxv1N2Abhn16OyODwkVkUM8FBjkk3DMSnjRrSqGE0qot5c//FwfUXMcZDs//zKKJd0fiLN9I4tJLS9tqczfpDjvKxRS5mKK3G7JPG96Umt96xNVRlYbxR/IKXH4jlWunj5wlCm/BRJ1ZDMey9T3j/kWcQT3JznCNJ4pDH1R0HXPyIwmuQA9oqCsxEvtd4wcO2AGjFcB41uwu0uoObaD7Kklt4yoyuSYQYNbUyveVYIfRwS2YwHzv4x9sYXaXITbMymdxQYxEnPVkGL/Ma3DwLiwRCZGTovnbPdiX6L+FcCHsKqkbAg0KdUZpOWiqDN+h5uFQc371+vVwqIemP5jGJk3TgaG/ikjVO8IcSm6uKU512Zv04zhkiTLJtHlEAEkJVSgcLbkdjt8ZZWRYh6ziDfClnu+QMdY+t1ya6E2Lz7KoJyxhcmyajQ2YxQG0ILMoVIG0d7Jq6F8yVND8go/0YzTCeEn/v2zHtivKNqn2tiL3RjrwbUr/QKkA0imXVlkUEoO6AXnxOzcBipA9VhNjm/HPGO2gVr0KuRgo4kCIiPj+0oIdUZjXKSxbMry6jlVzIG8LyM+aBPlKI+PKKN9oM47S2SibvVBA/fktQ2Z+2xNbgpjGciNtI1eWv3mhEHQs7cgkoZXmWiF0OuzM6NlpuitYyrYNYeGFnqY9gQ1jQIFGGA2JWkJbLboQc2M1JjIAFqayL5eNq+TiijxMKfeQ8IX2XB8IJOSgUCldFGjAXMw0A5TLaMzyKFuMCaztx+LJ3LxH0mka63Pxmb1eryixcqrRGi8v0Dh589MkBz7XwqK5DNFl6yUujmLbThI1SrlO6eKVCAYJ+Tk0MFS0Uz3xAsOnA4pjq2K+MqGbfK4ekveSoPP2lfYVJm9YablHBhbjx9xED+PBsuDx45jZNw1+irZ6Y8khyMyxpBgSI8uWL8CT6PhiHgZl+AMUpOQFxlxxPccu4y+zum54NjetxUYijS3j0xj18mvbdrf4SsVI8AVnb37mb2+rnoT5q+worHltP2/xZUu2YrsCWbVd8bgrnnTF065YNKxHp2z6oStK+4PgulwUlZaRkUXxWzGiH7Q+hOjh4SwiJ4A8lj5gjFD++qh/mHB4kYf48f6Mb/jIxuaRPivxQ0+a0t8kxriKe6E2ojqiCiZc2B2RfevCiR7zK52YtMt3jvOIv+loVrS58uO8zo2wceooAECKaCevFD+MZcAqwFdgqXgtT44joPIn4D5+RjylSZi8mqmSRiver6axzMS6vP1G0fL6BDqSx2MawtR2agUPFDrCPSHJadiPqZqYPGmZOMoU3e5CSVDTuTAO0KLIl1lkWywjeDFOmECIl8nCQ0U3J/S6ITjRYp0SxeMVgzjhYlyB4mco6Y34vRuvLRbZzgf1ihO0sbCvXJ3cr3QEJaNL84LO0IctTGyUPy8miPfxwzS3FBMRGQM0n00aQcW+5WkD5USnKaf0Ed9NaHrMmHNtJB/YPo0krztkCydv8Fq8uxq75dFW57hW2zqE6VrlaDfftXnkx7FeO4Kq3Bd3VWTAxYsO58F2ykewYX5VXZx4lRZqYsZOQW2LgnreEgX1YXTMDa0WXN6G+3uLPpUJXUCj6IlFxCW9BNXtfD0YaE8Hyr4dMS3cXmkpu/OGsqg+sHQ92QA0bt+MXk2myCOweaG5Gk6j9FhD91TSE590SCpDPiXz20Ewv/Ugc+gSehktjSmqNQcSzw7Uopm0t1iVnvgNneJho6dN3L9WXkri5bn4PG95k2bccEDcoNDr91msBRc5kjSC5ZQnQYZghWLStDTw4S6+ySK+wdc3aO7Mm0J/8UacZV5Md36byomi9fxY0XlMAVmwWDZN35qEuw4JUdyIfv2FNOHh/DbBgrxK6EPo+S8hizheF3w6wJuoXC/6+KTY002IU/MyY3ny1+dvz5db2g5ZPBnSOzykIFqv4crUBjoCu5Pa0hF90YSD1VGcz+eihghTR/sPw2tHU6bL+srQvFaIBz2pedtsQ0ybjKgp3UMFW9vQseXg6ERftOb9shOyAMr7GxTBEWVuE9Jzz8p2paHIdkUFffgOJ8urkSTtkrSQZdMHFkod1B1OZqTeoZul1V4PcZnsmL6VjiUr4CRMWyTMRXskjOP3jt+3ld+vqiZfqMTTG72APs1jCHHRDBPBnGNaER1spn+8gLXDbaaRhweFnv7oiW+JQ3LCRapnNkyODHRrDpTGAZ6RFDgr2wU+UFrQz0Ly7POTfJe7D1QAp0cXO8e+kkkuN5DCOJUzI9gBApuZhMudpu3upWCnxv7VcPKjeflxcM2KA2TIiXvNJ++S/ZgJS2aLTVnrUVna9WyGw1onSo0s/aKXz/NaKlQsQAZfHnsKXy4dWBxgmMa+nC1A9/xun9EJM3LgahO4zpoG1x22/6EbzrXUnaiyUXzCRq9DQKsQ0Lx8OVpZ3lT9KrukT68vcfxWTEnzroqTcGBqFZgalyenm4/roKq+aXN6lYkjdyh3A6riBByeWoWn81YIp4N9KZvq2pHblzsgdH6sWDpsq74svPaYCTgItQpCF01D6ET/4oY8qnNnfwfOKhaoODq+pURY1TY7B8JWgfCyaRDWvSn7EBGKZ5dHlmO6h4AxFwT0LsPzqml47tj1OiAIoizW4Xi3367QheMCFByFt4rCH7fA0X3oHm7F0J6qxSaqhOG4GJu3g7yfNE3enyiBio+rG6EHRSaUhR4cucVZGkdwXKyAo/JWUfnTFjBxkyHsK+a9fyMCMhGsyRDrDXo/KuTm6THVvQ8OjXFxL28RtTfu+r2f1M57ygs/P60KQw1J3i55+11D4EULxM2+sOU7s6ZrqUG8P5f1iPRCR+RtIvL2eGYPj73f7Yo9PFf15ALC1dK393tUDxy2Q02bUNO8w7SO7JN1HB2ZwX1yAeE6ShvshtgxM3I4axPOGnfb1pCftRlQdmjBg5PLClcr9lEWPnbQcB1q2oSaxr3BJRmJ24kxZQU9Tq/zu1W3psxqLxmEo+g2UXTjnt8ac2m3GxgfW7rm5KK99RVyKm9pfNTkHPrahL5nzYdQVsg+3w7gr17e6eQyv1WrmpUH71ccvEPOPSDn/wMGb9XveUcCAA==', 'AUDIT-DONNEES.json': 'H4sIAAAAAAAC/4VUyY7bOBC991cQPlu2tXhR+pYguQbIZTAYDAiaLEo0aFJhUVZ7Gv3vU5RsdzsOEBkSXMt7rJWvT4zNlIgw+8RmxarYZPkqy6vZPOnR90GOljbGDj8tl23fNMY1WkhYSL8koECIuGy8byws8ejtBA1wMmi8S2CthNalyMUqr/Wu1qJcifWmELu12spclVDn61VV3R3KtbHAsRXFejNyEALo2B3kSpeVBlnud/m2Ljfbrap0WYpS14XUMt9toS62daU0kFyrHDZaqYlb+t5FJL5XkkiOgHEZoLNn0uXr+QdtDMKhnQpTXC0nYQ3lTHndUJtH00doXl5JgzDvqKK+U39EbAuyvI3hKi/7I9xHnPxJ/GcUk8J3RvK84PzYv/9m83t7Wew4x4M5DPSa4bBHkf6Yw6+OVbXhvGmboWmG9Dbp+dVpvdpyLiIOPprB0PPgUKyJpTEN7vXgMcaokSQzG93+fSjZY0JFnnPedZ3tutjtreniPTb16Ddl2JacqzEkc5BEHaN7yJBctNHCaEPvhfVWcniRtlegbtyvV7RJulmacGpL9oeyZreOZvV7AAEETisBL0JG3gkTuOo7a2Tq/ej29nRJcuY8l8EjciSHyC+Q4I9d5EQ+AGEvA0KUMfTwcX9+9lTdmGZt9legOoBb4tnFFqKR7KtrrMGWoQQngvH4zD4L31tgxK8B0+YKy65JkIRMIFOAMpg9KLY/s67fJw4Ic+Z8ZJSbpY/0garHpHcnCDhBF+xbACdbJvrY+jDBBR2CUbg4Z727TAKoBfseyKcJomvP7ESxAS6m3T0KRylxigDA0R2UMtux24owBAuSKJ6ZFOGmp5AwBpMMLHqmTcDINuTsqCISkILvpQURsoTS5EgRs+sULNgPbyHrAmjzQlRHT5Ey4RTDjq7A7OBNCoV5Z89M+3Cp4jNVhDkYrkWNIjSQQhwbgOY/Yr5cdmPQdCTfA+GBj/s9ZXfrqKMynoDfrYsWFicrDQ84HG/pL1+yz39n1WL1fKu4El28NjCFPQ4HZRLpGJryp7f/Ab+BCTMBBgAA', 'CREDITS.txt': 'H4sIAAAAAAAC/z2RzVLDMAyE7zyFHqDYSX+Awglo4UIHhp44qrbimEntjOSE8vao7ZSLfdnVfiu95hw6gu3m/Q0sbPe5W2U3AckDO4K2lF7urW2HEGIKDToyLluPBYWK2HByW1HbBJjGKDEnaDw2zQxrrOplc7dscFbh4maKdwt/62o/o2W9qObzuYHnZ3j6grmpDGyJIHPUGOzgc/242qwBk4cee+J/EORDHE3mYHEndrqopqaezqrawDqFLkoL4ighxyyATPDDsRRKNlAixkL+AZ4wD9q4MCbpsCjwWdoPu+MA8lCQg7Yz8MKUXPsvJW/RY68/7H6htAQoEqVgKtBkho3yRjSXBM769ExNPJDodvZ5VOexk9NMckOJI4FQUkCnCul1vdffOaYjZsqQ8KQYsYv+BGrgQ5vx5TpddGenzsaYYFBYTIG8ufoDqFFY1tcBAAA=', 'source-smoldoc-en-bci.jsonl': '...', 'preparer-donnees.py': 'H4sIAAAAAAAC/51by3IbSXbd8yty1BFTgFUABerZZCAU3Xp0tKxHj6SOjgkOzUhUJYAUC1VQVhVFSlSE/8ErhzeO3rjVCy8c3s0S4cX8hr/E59zMKhRASt3jhUiikHnz5n2ce25Wyi6WhavUm7LI46Wu5pmdxHNdym9n4jq3SZGaVFc6ToosM0lli7zc+WkcBg9/wO9e9K5wJ7vOJHPj8G9gKr00d6L+wYsfX2+NLOpqWVfl7kLnqdXtyJ20SMrxIfUYZoVOy17ZV9PCqVLZXPV+2o3KRZFh0GBg8uNJYoccmkX9oTM6Pa7MWdXrD8tlZqvM5qbs9Y92vlKPncmTubKl0rnSZWnLSufVQNfVvHAmVZXTeZlp7kkVU1XNjVrWk8yWc3z5KJ/xr1jlRaW0mtdQuZFYFrVLzHDn8cvxh52bd/cPr32zdKtfSpVFq08wlInVG6MwtnJGzVaf8tUnpzOzwAMFA71Xi8IqU3HQVEO9hSlVak4L68rhtfjao7ysbdUImWmXQjHtpgWGZlpVq0/Z6tOpLUXvmqPeFLVRuj7D3/hxatPVp4KSfvCzRFJZOBji1CSynF5YWet5UeNDlsGt7Rq6VomFzgsd8288TmIuVNrFsruLt7XJ3tZRnaslDIJxZWmcSmtVmcWyVCYvzWKSGS7z1Kh3xpzAe2ncbhtmPKUw7GnmOFkt4AwsbLatcMrhjXqYC7Om3EQy1xV2R8spqDG12YKK0jaVkUeLugzynkTaLmglBMIlXb1dNGQvbOZVFsvAaFXXNOnqP3PjYhqcj1f/LnqI1bGudxa/6Jhq22Nds0CnJSLnU2KxR0f9MAFqnOKz3gypZgtqYnSdFPUSKysklnGQHoJpST28eRJD94jt8HC9FhxQBht0BSXzosRGSrgRsTlYrH6FqSCUDzL4pchTCrRqEUFFeHutz3z1c0V3wDZQtD518EVYPYtg1xyfIQy7hMVPMe8o3rl1Dznzg3FI4tyokdpXT7F6UixWn1KLv1yBbKssZNAFsEllsmBO6yBJIzUrIwHcStmDlCcIc814zyNTVoOEeVOq+yJ96QfqGf6uisWEwrATeJJ5seCeYD3GuzMLi+Dw3qQVbFZCIvV4oxcMRSakKybZ6pfFlhbcyyM4XbaTTwFIzQ6apdZb0a6yUwvfZ2VIVmXODAMnVimgqQnpmC6gp/ALSYKARjhqW0lmBkvDR4aRC38hhBnYUH71H4i31aeKY9sdMmw05fqY2rDfA5qto6GzqU1qDFQPjLepGBSOgkcRQomBqFKsXMoiNsdT0R1fMKINY/myhc50UskSQ1rLh1sGgAKc0NhTm1skKDRosiWmAfGx7FiSSYlVKf17ekisk9oSFcasfQX/8ZuO90zOWC11khQu/YwN2i3Sk3QCpmGP59gRBMGGOeEhZaqFSKd9N/Wb6sRmkkYdTZ9Rg6RrTTpXSwQ8DVJSw/poOhkamSnKYVX6vUIybLEG2kpQKAzxYIzgKytXU786ZxkJdS6tnZ5klz0SMmcYtu9l0LsBRfJOKcpqyYAr01WQO6PRUspmlCPmZnCjR+yKrsEf3TT2KOP3LvBwZwR4eIYaua/+VDeZTPTRztlTQ9itNFG/Qkgjvbkgca9ixkyw64pRj+o+IxByZZcjo0tvL6r4GuEAa+yrF7X11VE2/4SeNIvF6pfc+BrHxAKu6Zx5motnbC5+FPUeNboZb61TDbOs/sriQzak7nfX+sHU1WD1K0jBUD1d6097I0mYkHQlbbe0q1+SBvtSOyVK8Dk37lg13VBJIL2BdZarX1yjQgirUJVNRZBogGqt9bege2oivkcFzUFl1ENbDshKwmZSW21tfcNqTwRxpgYJDkXA+qz4bXTjhjguJwbBrCwPbVljmP/Pv8DTE2dLlqyciy0KjwLPaLgluBtN4TNsIT6vaMtQC1NBaAmlNqKp12aQlE1mcRU64ImEb+k5BLws4aK7NZrbhc2Iez7SBXwEYhnscKyHKCmTQy8S7ESQcAnjeixjiUnr8PUSIc8VaU3BCkQkybOaG8HhrhSiUQGGELPQ0nYSiZTt63VwSlUHzbCHH6lmvWQAxSqRHafMnvgaIlqmd7aM2XCtEANJfC+1s38Jlppm9Ygi6V8VVkazqAgUYn1DIg1cKgufBT/UrLYeuUVB1HgMgLt+rWIPoggvxi04JgmXJwVAi4Bk53BEB+aAa25dq33Ek3MXpNKusZnkqo//zX1CyTdrm77BZNFkU0qHt4YFKXejwIGOgqmo1zWeaS/a7xkzJIGDXKlJqaxYYfxlwzJduiZlbC1MFdaUzOk4/QEhBGEGsUAPjniICZJ8kl1kTa8SAad99VCSYUH7P9HlgqkD9HiCAF2iaxHkkJ6npRKJnq4+iQnDcI/6NFabpDQRAbqJN014X9ZC4EyAHiA661sazYmdfsGWDbaSGnsZsYG2qc8bGe1XFvggsbLShRofpU3otbu6IsE94TA+ububqeg3V/qat0TXZdG+rnVqkZQ0j2O1zYZrgKibKFr9VRoM6iKthdQ9J34iPfN50W7lxRwJnm/oe8muTYaHBPBE3P3+vfmqzF2hU5vBN3NUziLk8zOhvEhNcux6ITiiIQb+IsRLfWkUsfmbGjUXo2kokv2mYhM3cyWKrSkMEif0qogAqegS86YMZYrJWq1+RrT6yg+GFjC0He+Rk3al+p4ThqrTtWJDPVHhhe5tWhNp6Es/5UCpxBWeennMrsT1yEHPQC8HufbdbpNgZSVACkzOCweDSQ26VCm2YLNVVcqnT/1IT8AhYxWAmjsEdpC7QQ+Yx5MumC/R2cIMBFKZyqPbSOXH+BaOekZiWdSp7SQk6LGziQl9HbISVN56CrUwNQzkyYof23a0WIvdM+mm562EboQG+JWomxhhZowz+arl2Ain0wCvcHvuedkl9nUqZwXkcNZxGYkgzxIqNkywxlrtjAUdTl562mkavSWwr9o50WPdj1kIlWMNJ9bjek627zvVcKZQzVef0H66QJyaYwu6GCWEQNtwicicLR1SQEoQ2KoV0kS5UOUU8fYllVtxQnu9Lr6IBEqgAlkXKh7VlQ39+pYxvfk9CtUZ+nY5eHg/EMM2BQrSmw6rY06vqXz6bVs+jRiI6fZaQuVp5i8tVXrip3OhgWSCdLPlIYMUkoRdFpq3QDZymFa3JzmeR/mGgWU+5BEEgm2SiaKT8ghfE63+OxNqkCCC/ba0choLFEJFCLeFszNf2eY5+wtJYlH4SoMEzkkqg5qlM0o3vug3VTAoD3BaMFjaXJlTannZa7Iaa2nrKGQTVKNBkm0Lfy62jSwBzdq5RtrQUIMynQt9o6PxVBhozpzXbdMaYINnjT82IDNYSBDTZVLpjd8kSr80iKa2YoCJLtnDUzjtVVRG+F6dSe17pX2fYAZyJKB0WkDXREhAR3I4CEI3oJf+vKlpB7fOtyjzu452sntCLkLEsg2EF9eM8U2IEnKE1c8zE3uuzOARsgxiWEsq0tQom9oZng+UxZIFEBEBq7eHdanxk3hoh6otcSYQ4rvdziY7J0UdSwrhW5DwIQobDwNC2mIrB4oQEOCBJ4KXt/toribYaMPKWfDUH7ZV9Id0qBrIGyAGAgZh7ouFqC1W0AvkhWu31xqFKVXxSdKcPfoTX8EJPEJFojAjZyStNElk2OUKY4RWUs5Hs8sys02ZQ/V9FlpTaogYLa90ei6HMaGQyrFOcw7DR81UcpXtExHvZhUQyFeB1MwLfzz+p1qTotTyRRm+iPlgGs7m0MOfkVp76JS2KZD/MhzdnHoobvuM3x0hXVO11WrLZl+ICelOlkuWjMCwzUa/QLqs21cGTX97hWYdIUUthOep3kAFvqMAGvMMNzQd/vRAAMCFCRuZ3x0Xkttdzm7C0B7PZzZ32MDkEoxkgF1WESn/Vq8gBDqfEtXuN9sahActThAjurx5UaBAh8Zl+/w74Cc1lowKwfTmMmFcROdq3XhditUv9drd9yhYS1rttpEQJXxbRGpJXVu3SaxAUOdYPrCGMli+oTOh6wk2Y0kMnbM46mHDMlHypnBFAjzioIzl0pYNZmT6be3ZcMiExHSbHNKPcD6Jujp5w5d5U/HD0077znnC64T7sgM8JVVfb10ORMLJphzDlJW0Rq2T2vc2ABxWDokgdggsbc22eSi9NGtOzq2wI6IJwZzBnlFELcpZ5c9PO4ti48iPcBrwGn1AaXm6xXARC9GVPDUm2e6eTEBNNgQ0OQ0VVGg1CMbXPvplU0IGhSZJA9Td4tTmfMnAXkYASxiFaQ5um3MOm3723KQJKDISSX9FiKqdnnkgKGtEWPXluNjKn3Ca0IUr39JIhZXTa27Kv5koWUcvp0I4b+imarefXSMej1baQ6Kjjzuvf3z5/NX4A1/sHN44ig9H+LcX38TPW/h3O76Dn3fje/HX8Ui+HsWjvaOjmCe96/H4187Al3Kc2JUW+2/iw0vS+IPTR5w/uiWTqcooFqHxlhIcdCMWJeIwJR5R8OhOPLobj+7xz68phY3a1RuK7+I3B/pxYGaf+/4KJTFjr916o2Mss8K2Wu3iEQWObscjqu+1g8Ffvnj6aAwXM9yRFj0X/VPv/n57ps+VLl7BtwpPT+r8Ih/+7b+KvH/xrLAXz+yFP8q9CF3yhe9tLzrU9+IfC0RMrt6hu7yvp/Xf/u1im9ledGLnolujLrqRefFc57nOoYaaZLp//+KbvM7vL/yTSme1Vic679/v/6X8h338i2Ls6vv+TmqmKskMxpX9fWcqaKa46WFZT3pRFJdDdPF22euHoRqMuTM0UtHwDbKt17nEMGS3j+29N73o+eMHkNEfJro00yJLe7xLsMzQfvSi//3nf43ia9G1cKWAS7jiXTk+PJKldJr2TuITm6exTc9itDWLZRWDer8zLvZ3A47xpSVsyoHaWdV8Hj8vctPf31HpmFceDk+ODmSJcXQKvVIpHZECLp+Mx7fuKZMBQCMAZuUf8jYEIpfBcfuo+RY1II92FDUcgiIDRntYq+rZdDy91tyb+JAeRjaNjj4OPlBx/ILqH6816mJITdwZh2FxpcuTsWwRjRIWGEep1Vkxqw2sJhrLz7D58YYNxleaYvwbltn6DMapDo8aGcbfxhgfQr/SJWV0dGiP5IaIpU02RbezJgjTzMikys1+5yQGSmngkVlRzDKzSwtGzZfOeB40jqapnk5v6pG+Mfp6eu/rqb55Q9++s6fv3U7vJqP0pvl6dPvGrVu3ojiDeODtOHrwYPDtnwe3hjeieCoXSY59ezuO2ssp23dSwi2UA7mF4u+ghEAxaRT7/bVS1pdXKu1mbAaKpTXpAUIjk2Zjas/4ts4silM+XqBrtQMeAckhvZ8E9oSsgYzJuSqXSIcyinM5I5HNm3cmPZ6cH/sLPOPHGjGI9KBRT+KpXNd5/HIIJFiUvc04lxsXbCby3rTsj8f8o/Vmn+5GzN8Z7XgHxVOKMjmiEjzYcMq+z7uoMREi0cYeIab98EfX0/2YP7y8PO7peNLflPke6CF1a4h9Q/Ihpmx9PhztH/W5D/WVepF7npiZita0ECtIY2nSSovR3s1Z6Um5+JYVbjNkhktwP6UTB0RVTaaVQwjl/LEsSQstaWC0amM+PswHoyMmfe7zHNijQJNKBGaDbGHvJbe5DmuNLauQlr3ogc8q1vbo+m/MbNbvX4/+4lGoeRKgJsI3gv4zL4/qYLHgF8LnOYKlAcSt1TZcs1500o8ncbPOdSj/lfoOhBosKiVbLuWl2T7iH7Z8p76ViKfJZyYXL4IWe7ghibbyykVp8K8k07zj5fNWNUE+3AnR8Da2discDnvkyEKdByS+m/dvyJRRrvvx1iheHKrChSoZQxKwPSpbX3+SMSj1MuYbdDky4Mr7SFV7H0km3emKxZL+NKq5IEGK2R19N4yWVlXWCGcO4T0UOqtK3kHI6NFR/4hhTlfevNv6MpoFVwyi66i2vbwfv73kV+b3zbtHV7sXVqal/W08897wPaAy2mU8MJ6jNolv+ZpAn4MfI7/4apaulrpm8xkL21A9xxMOL2ogFMcSEJ3hwWEjzfBSieQacgtDWa6phyDJ4ZVXD2/t8iYNTJYNEII8dggXC+PPDKddLMz5e0bf3kUPVdkUfHpr/BEsDQWHTExU689ce9y4OTn9/FXH/g4SruKxZjr+4GmQA5x4SY6SsBaz2QFtOS86+sM40AYZ42lF5NMWm/H5BK6wUxqTH9OUn5f7ZQFz89nJ5Cv/P61ODKDt8OjAGTax/It6jlG0e74SOV/eHSzS4zLxiTkfZ3oxSdFm7vfWK46bFWPneY9gPUaXa6UbDZBP7aNGl/7HA8bbuPe7BgOY4cOS7AGWOw7hfIxi7DK99AwPS/+xdWeAXWRDO7hByuz8mJuOPmdBHpGJsMaFHVky5Mrlr7DMWhJ9GaRIqh1z68dpjQkJq7EUDFY6Gh+r+rEkvDsiW7a+753WENUPtPp+MD5hh2PwQH5/7B+QFNq8hgRKHBKeuET/gDHQCHHe6yjwEjK9xqddTg0wbB8Ln+aDzvfNU6amRJ85vzr+4miLMks2K607YW7Ou4FOVbeNqz8eTCZ/14zJR67iOZRcSdZ/nEyE1TC6Y/nYiX6KIJ77CYC4LlrnBJGTdZY9fsnVAlHzvhiPufj2TvubKM8sXsf3GvTXUwO5xsydFz++Hi5OUut65gxk97g4Gb92tenv9PDNbttebGNrf/jOodp54ItC5RHETGtUyp6LUQFrZ451mVgbCKknMBvGQPZp9KzV+ENE9hztR3s39u4MRjcGo1utU/F0XlXLcn93d17PQKpnUzBg9Ni7oSsodze6gqjpBzDx9zcEjXHQK5vjcq73bt+J9sO1+6H/3PvNO++T80rAfzg3Z6mdIXh7/ThKUK6rMtqXBrBzZX/4gF8Y18G+69FudB0f2ekF53WMFUctUY32P5T7AU0/XBUVXw7c8mOnph1elZsh944+xgCWJKvBN6KAE3GUF8dCnI9F4HFAHknKYx95a032GVCted/WWKI6h2t+crxGlu+W53k1N5VNmr5K8UWXBpUvDxpqyZeURl6l66zbjZU8wEtNmTg78d1R0265OJARTHCGd0rxPYCLx6h+6rD5jwPtfz3A9Lbn4yFg29QN1QuHMTOnl/Nz9HoObGMIC4FYYDvHWB44iLjEru6tewmgYyYV40Al2rXP5SDTWSkloMVTMKNK3WnJMO9J14QFN+CsqU38bYXgg6F6Cd478H2j7xqxQ5YD6QsHzETyMwTluXjYW/Bgi6s3baU3fmnfQzJzwITYPJ4YTDbHDd9rvBiazk6c7Etyx1FoqGGCtqM+aC2c6mXVOCxv+X/ToEUfA9x88+PD718PHr54/vzRo1eSWZtI00EYgY0rUCYGuPHIZK/fYNiDl48g9dWwOqu2cOs7QQ316tmLp2pXvUJiPyySuNHu78CcWDWQg/5LNDtcgxByeqgePFDf/lnBJkP1ypjwBh2Oe/nom4fPHolRlmCnrl1VuzN7OizcbFdPyt292zf2hqO9mzdGw8tZooDL6l3IprYDa3NnM11c97/ThChoM6Ft4tNdcZnPCf4XnPVZCGPqmRDqYbPClScZsic5005qxsw6wDuRSjUZmBJVah1VvLXHVj64QmIr8bIRMUhN3lWaIWZRUci00WL+vcHxf0NPVkpmNQAA'}
SHAS={'code/mandia_model.py': '629583e3a5a5e1c3ea418f30b3d2fab7d9e05e07dba5adc4e7563436ff957d96', 'code/mandia_seq2seq.py': '6d1b84f0fba29e716741c11cff71bca8b622c9d850feb392f9d77c7b9ed2f1af', 'code/mandia_stage4.py': '14efa69d555ae9a9bae10302281d2cb0e261498d317ee482d587ee07fa433086', 'code/mandia_stage5.py': '023dded2cfec26496f667d70bb73064a8e87c8f934e16f87b67ff9454e81d2d7', 'code/mandia_stage6.py': '4a8ef3a0b5fe609add76bd87c390a9907c6393dbabaca863d4ca3c6d6555f4c1', 'dialogues-prepare.jsonl': '9375c6a0918a5bf8056cecd42df0c8aff90c3439220bc45bfe10e8cc700a2184', 'AUDIT-DONNEES.json': 'b60efe3a7bc4bdff47573c0df521223deb204f1d67284b518b1a56a40737136a', 'CREDITS.txt': '52182b758b69073a8671ec99d20e33fe59cdac7a7f7833a1d77bb64fecfd8f46', 'source-smoldoc-en-bci.jsonl': 'fa56efac8e1df34fec3b81793677d4f33a3f92cfc187e92794dfe2cf9d1e6fdd', 'preparer-donnees.py': '7d7b61dcd2e5da10be2fad769e457fb978ad133b647f7b77bcc1d98e72f8de6f'}
for name,encoded in ASSETS.items():
 data=gzip.decompress(base64.b64decode(encoded));assert hashlib.sha256(data).hexdigest()==SHAS[name]
 destination=ROOT/name;destination.parent.mkdir(parents=True,exist_ok=True)
 if destination.exists():assert hashlib.sha256(destination.read_bytes()).hexdigest()==SHAS[name], 'Source changed: preserve this run and use another directory.'
 else:destination.write_bytes(data)
try:
 import tokenizers,sacrebleu
except ImportError:
 subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers==0.22.1','sacrebleu==2.5.1'])
sys.path.insert(0,str(ROOT/'code'))
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'Select a GPU supporting bf16'
parent=ROOT.parent/'mandia-compact-s2s-etape5/model.pt'
assert parent.exists(), 'Reconnect the same Google Drive account containing the stage5 checkpoint'
assert hashlib.sha256(parent.read_bytes()).hexdigest()=='b19182b18a271dc7a6b36abc60ba07dfbb5848e3f020f7c61e2bf3eedf46042e'
print('MANDIA_ETAPE6_PRET',torch.cuda.get_device_name(0),'Drive:',ROOT,'Parent checkpoint SHA verified. Heavy files remain in Drive.')

# Freeze replay datasets as well as the selected parent weights.
PARENT_DATA_SHAS={'mandia-compact-s2s-etape4/parallel-prepare.jsonl': 'dc2bfd9ba783c5edec159ec35bfbb66206598520bef31f960da3ad92cc240cbd', 'mandia-compact-s2s-etape4/replies-prepare.jsonl': '29334943cd507b8ffa9385b530b9625aec82c0741aeade3575a30c7df7e2caf1', 'mandia-compact-s2s-etape5/quotidien-prepare.jsonl': '370d3bb631a4ad4765fbe3b15e0f6019c0470ce03c687f87ac71f6fda6b3048c', 'mandia-compact-s2s-etape4/data/waxal-baoule-text.jsonl': '4f7730d32d3324d2cc819f33d7633b2d59e01f81c8e2f38651d0377050ec69d4'}
for relative,expected in PARENT_DATA_SHAS.items():
 assert hashlib.sha256((ROOT.parent/relative).read_bytes()).hexdigest()==expected, "Replay dataset changed: "+relative
print("REPLAY_DATA_HASHES_VERIFIED",len(PARENT_DATA_SHAS))


In [ ]:
import mandia_stage6
report = mandia_stage6.main()


In [ ]:
from IPython.display import display,Markdown
import pathlib,json
ROOT=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape6')
report=json.loads((ROOT/'rapport-etape6.json').read_text())
chosen=report['selected_by_validation'];before=report['before'];after=report['arms'][chosen]['test'];contrast=report['arms']['source_contrastive']['test']
lines=['## Étape 6 terminée — 2 × 800 étapes sur L4',f'Candidat retenu sur validation : **{chosen}, étape 100**. Architecture conservée : 13,78 M de paramètres.','','| Mesure test | Avant | CE retenue | Contraste |','|---|---:|---:|---:|']
for task in ['translate','reply']:
 values=[x['generation'][task]['chrF_pp'] for x in [before,after,contrast]]
 lines.append('| chrF++ '+task+' | '+' | '.join(f'{v:.2f}' for v in values)+' |')
for label,values in [('Perte nouveau test',[x['new_test']['loss'] for x in [before,after,contrast]]),('Perte ancien test étape 4',[x['old_test']['selection_score'] for x in [before,after,contrast]]),('Perte ancien test étape 5',[x['prior_test']['loss'] for x in [before,after,contrast]])]:
 lines.append('| '+label+' | '+' | '.join(f'{v:.3f}' for v in values)+' |')
lines+=['','**Résultat :** progrès modestes sur le nouveau test ; léger recul sur les anciens tests. Le contraste ne gagne pas la sélection de validation. Les 14 réponses simples apprises restent exactes.','', '**Limites :** 25 traductions et 15 réponses issues des mêmes 2 documents, pas 40 références indépendantes. 0 traduction/réponse exacte. Réponses génériques et erreurs de sens encore fréquentes ; aucune validation native. Français traduit/adapté par assistant, baoulé publié dans SMOL.','', '**Drive :** checkpoints complets des deux essais à étape 800 et meilleurs poids à étape 100. Le modèle reste un candidat de recherche. La dernière cellule permet un essai en français ; exécuter seulement la préparation puis cette cellule pour tester une session déjà entraînée.']
display(Markdown('\n'.join(lines)))


In [ ]:
#@title Essayer MandiaCompact : français → baoulé
message_fr = "Bonjour !" #@param {type:"string"}
mode = "Répondre" #@param ["Répondre", "Dialogue avec contexte", "Traduire"]
contexte_fr = "" #@param {type:"string"}

import pathlib,json,sys,time
import torch
from tokenizers import Tokenizer
_root=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape6')
if not _root.exists():raise RuntimeError('Exécuter la préparation et connecter le même Drive.')
sys.path.insert(0,str(_root/'code'))
import mandia_stage4 as _s4
from mandia_seq2seq import MandiaS2S,S2SConfig,generate
_report_path=_root/'rapport-etape6.json'
if not _report_path.exists():raise RuntimeError('Attendre la fin de la comparaison et la sauvegarde du rapport.')
_result=json.loads(_report_path.read_text());_selected=_result['selected_by_validation']
_path=_root/_selected/'model.pt'
_dev=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu');_s4.DEVICE=_dev
if '_loaded_model_path' not in globals() or _loaded_model_path!=str(_path) or next(_model.parameters()).device!=_dev:
 _receipt=json.loads((_root/'SAUVEGARDE-VERIFIEE.json').read_text())
 _expected=next(f['sha256'] for f in _receipt['files'] if f['path']==_selected+'/model.pt')
 assert _s4.sha(_path)==_expected, 'Le fichier de poids a changé depuis la sauvegarde vérifiée.'
 _saved=torch.load(_path,map_location='cpu',weights_only=True)
 assert _saved['tokenizer_sha256']==_s4.sha(_s4.ROOT/'tokenizer.json')
 _tok=Tokenizer.from_file(str(_s4.ROOT/'tokenizer.json'))
 _model=MandiaS2S(S2SConfig(**_saved['config'])).to(_dev)
 _model.load_state_dict(_saved['state_dict']);_model.eval();del _saved
 _loaded_model_path=str(_path)
_prompt=message_fr.strip()
if not _prompt:raise ValueError('Écrire une phrase en français.')
if mode=='Dialogue avec contexte':_prompt=('Contexte : '+contexte_fr.strip()+'\n' if contexte_fr.strip() else '')+'Message : '+_prompt
_source=[3 if mode=='Traduire' else 4]+_tok.encode(_prompt).ids+[2]
if len(_source)>160:raise ValueError('Raccourcir le message ou le contexte.')
if _dev.type=='cuda':torch.cuda.synchronize()
_start=time.perf_counter()
with torch.inference_mode(),_s4.context():_ids,_=generate(_model,_source,1,2,96)
if _dev.type=='cuda':torch.cuda.synchronize()
print('Français :',message_fr)
print('Baoulé :',_tok.decode(_ids))
print('Temps : %.2f s'%(time.perf_counter()-_start))
print('Réponse expérimentale : erreurs de sens encore fréquentes. Texte uniquement.')
